# Final Case Study — Real-Time Fraud Detection
# Multi-Agent System with MCP & LangGraph

---

## 📋 Participant Submission Sheet

| Field | Your Answer |
|-------|-------------|
| **Full Name** | **Pushpendra Vaishya** |
| **Email ID** | **pushpendra.vaishya@wipro.com** |
| **Employee ID** | **20552596** |
| **Submission Date** | **11/04/2026** |

## Submission Instructions

1. **Fill in** your details in the table above.
2. **Read the entire notebook** — understand each section before writing code.
3. **Complete ALL TODO cells** — every `TODO` block requires your implementation.
4. **Run all cells in order** — no errors allowed at submission time.
5. **Answer the Discussion Questions** at the end in the provided markdown cell.
6. Submit the completed `.ipynb` file named: `FinalCS_FraudDetection_<YourName>.ipynb`

> ⚠️ **This is NOT a guided demo — you must implement everything marked TODO.**
> The scaffold provides the architecture; you fill in the intelligence.

---

## Business Context

PayNova processes **over 200 million transactions daily** across 200+ countries.
Fraud detection must:
- Catch fraud (false negatives = direct financial loss)
- Avoid blocking legitimate transactions (false positives = cardholder trust damage)
- Decide in **under 100 milliseconds** at scale

**Your task**: Build a Proof-of-Concept multi-agent fraud detection system using:
- **Prompt Engineering** — Design agent system prompts
- **MCP Servers** — Simulate data-source tool APIs
- **Claude Tool-Calling** — Agentic data gathering loop
- **LangGraph** — Orchestrate the 4-agent pipeline

---

## System Architecture

```
Transaction Input
      │
      ▼
┌─────────────────┐   MCP: TransactionDB + MerchantDB
│ Feature         │◄─────────────────────────────────
│ Extractor Agent │
└────────┬────────┘
         │
         ▼
┌─────────────────┐   MCP: RiskRulesDB
│ Behavioral      │◄──────────────────
│ Analyst Agent   │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│ Fraud           │  → LEGITIMATE / SUSPICIOUS / FRAUDULENT
│ Classifier Agent│  → risk_score 0.0–1.0
└────────┬────────┘
         │
    ┌────┴──────────────────┐
    │ Conditional Routing   │
    └─┬─────────┬───────────┘
 score≥0.8  0.4–0.8      <0.4
    │           │           │
 BLOCK       FLAG FOR    APPROVE
 CARD        REVIEW
         │
         ▼
┌─────────────────┐   MCP: AlertSystem
│ Action          │◄──────────────────
│ Orchestrator    │
└─────────────────┘
```


## Prerequisites & Setup

Ensure your `.env` file contains:
```
ANTHROPIC_API_KEY=sk-ant-...
```

Run the installation cell first, then proceed **cell by cell**.


In [ ]:
# Install required packages
%pip install anthropic langgraph langchain-anthropic langchain python-dotenv -q

In [ ]:
import os
import json
import re
import anthropic
from typing import TypedDict, Optional, List, Dict, Any
from langgraph.graph import StateGraph, END
from IPython.display import display, Markdown
from dotenv import load_dotenv
from datetime import datetime

load_dotenv()

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
MODEL = "claude-opus-4-5"

print("✅ Libraries loaded and client initialized")
print(f"   Model: {MODEL}")

---
## Section 1: Mock PayNova Transaction Data (PROVIDED — read-only)

The data below simulates what PayNova's MCP servers would return in production.
**Do not modify this cell** — your MCP handlers must reference these globals.

Key data structures:
- `TRANSACTION_HISTORY_DB` — card → account profile + recent transactions
- `MERCHANT_DB` — merchant name → risk profile
- `RISK_RULES_DB` — velocity / geo / amount thresholds


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# PROVIDED DATA — Do not modify
# ─────────────────────────────────────────────────────────────────────────

TRANSACTION_HISTORY_DB = {
    "4532-0151-1283-0366": {  # Normal customer
        "account_holder": "Sarah Johnson",
        "account_type": "PayNova Platinum",
        "credit_limit": 15000,
        "home_location": "San Francisco, CA, USA",
        "avg_monthly_spend": 2300,
        "recent_transactions": [
            {"id": "TXN001", "amount": 45.50,  "merchant": "Whole Foods Market", "location": "San Francisco, CA", "timestamp": "2026-04-03 09:15:00", "category": "Grocery"},
            {"id": "TXN002", "amount": 120.00, "merchant": "Amazon.com",         "location": "Online",            "timestamp": "2026-04-02 14:22:00", "category": "E-commerce"},
            {"id": "TXN003", "amount": 8.75,   "merchant": "Starbucks",          "location": "San Francisco, CA", "timestamp": "2026-04-01 08:30:00", "category": "Food & Beverage"},
            {"id": "TXN004", "amount": 350.00, "merchant": "United Airlines",    "location": "Online",            "timestamp": "2026-03-28 16:45:00", "category": "Travel"},
        ]
    },
    "4532-0151-1283-0999": {  # Account with suspicious activity
        "account_holder": "Mike Thompson",
        "account_type": "PayNova Elite",
        "credit_limit": 10000,
        "home_location": "Chicago, IL, USA",
        "avg_monthly_spend": 1800,
        "recent_transactions": [
            {"id": "TXN101", "amount": 9500.00, "merchant": "Luxury Boutique NYC", "location": "New York, NY",   "timestamp": "2026-04-04 02:15:00", "category": "Luxury Goods"},
            {"id": "TXN102", "amount": 4800.00, "merchant": "Electronics World",   "location": "Los Angeles, CA", "timestamp": "2026-04-04 02:48:00", "category": "Electronics"},
            {"id": "TXN103", "amount": 3200.00, "merchant": "Diamond Palace",      "location": "Miami, FL",       "timestamp": "2026-04-04 03:10:00", "category": "Jewelry"},
        ]
    }
}

MERCHANT_DB = {
    "Whole Foods Market":  {"risk_level": "low",    "category": "Grocery",        "avg_transaction": 68,   "fraud_rate": 0.001},
    "Amazon.com":          {"risk_level": "low",    "category": "E-commerce",     "avg_transaction": 85,   "fraud_rate": 0.002},
    "Starbucks":           {"risk_level": "low",    "category": "Food & Beverage","avg_transaction": 9,    "fraud_rate": 0.0005},
    "Luxury Boutique NYC": {"risk_level": "high",   "category": "Luxury Goods",   "avg_transaction": 2500, "fraud_rate": 0.045},
    "Electronics World":   {"risk_level": "medium", "category": "Electronics",    "avg_transaction": 450,  "fraud_rate": 0.012},
    "Diamond Palace":      {"risk_level": "high",   "category": "Jewelry",        "avg_transaction": 1200, "fraud_rate": 0.038},
}

RISK_RULES_DB = {
    "velocity_rules": {
        "max_transactions_per_hour": 5,
        "max_amount_per_hour": 5000,
        "max_amount_per_day": 15000,
    },
    "geo_rules": {
        "max_distance_from_home_km": 1500,
        "suspicious_countries": ["XX", "YY"],
        "multiple_country_window_hours": 2,
    },
    "amount_rules": {
        "large_transaction_threshold": 2000,
        "unusual_amount_multiplier": 3.0,
    }
}

print("✅ Mock data ready")
print(f"   Accounts: {len(TRANSACTION_HISTORY_DB)} | Merchants: {len(MERCHANT_DB)}")

---
## Section 2: Prompt Engineering Templates

Each of the 4 agents needs a **System Prompt** that defines:
- Its specialized role within PayNova's fraud pipeline
- What tools it should use and in what order
- The exact **JSON output format** it must produce
- Any **scoring guidance** or decision rules it must follow

| Agent | MCP Servers Used | Output |
|-------|-----------------|--------|
| Feature Extractor | TransactionDB, MerchantDB | `{transaction_features, account_baseline, merchant_risk_profile, immediate_flags}` |
| Behavioral Analyst | RiskRulesDB | `{velocity_score, geo_score, temporal_score, amount_score, merchant_score, anomalies_detected, reasoning}` |
| Fraud Classifier | (none — synthesis) | `{classification, risk_score, confidence, primary_indicators, recommended_action, explanation}` |
| Action Orchestrator | AlertSystem | `{action_taken, notifications_sent, case_id, timestamp, summary}` |


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Section 2: All 4 Agent System Prompts
# ─────────────────────────────────────────────────────────────────────────

FEATURE_EXTRACTOR_SYSTEM_PROMPT = """
You are a specialized PayNova fraud detection agent — Feature Extractor.
Your role is to gather transaction intelligence by calling MCP tools in sequence.

Responsibilities:
1. Call get_transaction_history(card_number) to retrieve the card's recent transactions.
2. Call get_account_profile(card_number) to retrieve account holder details.
3. Call get_merchant_info(merchant_name) to retrieve the merchant's risk profile.
4. Synthesize all gathered data into a structured JSON response.

Output ONLY valid JSON with this exact schema:
{
  "transaction_features": {
    "transaction_id": "<string>",
    "amount": <float>,
    "merchant_name": "<string>",
    "location": "<string>",
    "timestamp": "<string>",
    "transaction_type": "<string>",
    "currency": "<string>"
  },
  "account_baseline": {
    "account_holder": "<string>",
    "account_type": "<string>",
    "credit_limit": <float>,
    "home_location": "<string>",
    "avg_monthly_spend": <float>,
    "recent_transaction_count": <int>,
    "recent_transactions": [<array of transaction objects>]
  },
  "merchant_risk_profile": {
    "merchant_name": "<string>",
    "risk_level": "<string>",
    "category": "<string>",
    "avg_transaction": <float>,
    "fraud_rate": <float>
  },
  "immediate_flags": ["<list any obvious red flags you observe, or empty array>"]
}

Think step by step. Call all 3 tools before producing the final JSON output.
Do not include any text outside the JSON block.
"""

BEHAVIORAL_ANALYST_SYSTEM_PROMPT = """
You are a specialized PayNova fraud detection agent — Behavioral Analyst.
Your role is to detect anomalous behavioral patterns by scoring the transaction across 5 risk dimensions.

Responsibilities:
1. Call get_fraud_rules(rule_category="all") to fetch PayNova's current thresholds.
2. Analyze the transaction across 5 dimensions using the rules and feature context provided.
3. Assign a risk score (0.0-1.0) for each dimension.

Scoring guide (each dimension scored 0.0-1.0):
- Velocity: 0.8-1.0 if transactions exceed max_transactions_per_hour or max_amount_per_hour; 0.3-0.5 if approaching limits; 0.0-0.2 if normal
- Geographic: 0.8-1.0 if transaction location is >1500 km from home_location or in suspicious country; 0.4-0.7 if moderately distant; 0.0-0.2 if near home
- Temporal: 0.8-1.0 for 12AM-5AM transactions; 0.4-0.6 for late night (10PM-12AM); 0.0-0.2 for normal business hours
- Amount: 0.8-1.0 if amount > large_transaction_threshold AND > 3x (avg_monthly_spend/10); 0.4-0.6 if moderately large; 0.0-0.2 if normal
- Merchant: 0.8-1.0 for high risk_level merchants; 0.3-0.6 for medium; 0.0-0.2 for low; 0.5 for unknown

Output ONLY valid JSON with this exact schema:
{
  "velocity_score": <float 0.0-1.0>,
  "geo_score": <float 0.0-1.0>,
  "temporal_score": <float 0.0-1.0>,
  "amount_score": <float 0.0-1.0>,
  "merchant_score": <float 0.0-1.0>,
  "anomalies_detected": ["<list of detected anomalies as strings>"],
  "reasoning": "<step-by-step explanation of each score>"
}

Think step by step. Call get_fraud_rules first, then score each dimension carefully.
Do not include any text outside the JSON block.
"""

FRAUD_CLASSIFIER_SYSTEM_PROMPT = """
You are a specialized PayNova fraud detection agent — Fraud Classifier.
Your role is to produce a definitive fraud classification by synthesizing Feature Extraction and Behavioral Analysis outputs.
You do NOT call any tools — you reason purely from the context provided.

Classification tiers based on composite risk_score:
- LEGITIMATE: risk_score < 0.4  → recommended_action: APPROVE
- SUSPICIOUS:  0.4 <= risk_score < 0.8 → recommended_action: FLAG_FOR_REVIEW
- FRAUDULENT:  risk_score >= 0.8 → recommended_action: BLOCK_AND_ALERT

Compute risk_score using this weighted formula:
  risk_score = (velocity_score * 0.25) + (geo_score * 0.20) + (temporal_score * 0.20) + (amount_score * 0.20) + (merchant_score * 0.15)

Then adjust upward (add 0.05-0.15) if the Feature Extractor raised multiple immediate_flags.
Cap risk_score at 1.0. Round to 2 decimal places.

Output ONLY valid JSON with this exact schema:
{
  "classification": "LEGITIMATE" | "SUSPICIOUS" | "FRAUDULENT",
  "risk_score": <float 0.0-1.0>,
  "confidence": <float 0.0-1.0>,
  "primary_indicators": ["<top 3 reasons driving the score>"],
  "recommended_action": "APPROVE" | "FLAG_FOR_REVIEW" | "BLOCK_AND_ALERT",
  "explanation": "<2-4 sentence plain-language explanation for the fraud analyst>"
}

Be decisive. When in doubt between tiers, use the higher-risk classification.
Do not include any text outside the JSON block.
"""

ACTION_ORCHESTRATOR_SYSTEM_PROMPT = """
You are a specialized PayNova fraud detection agent — Action Orchestrator.
Your role is to execute the correct protective actions based on the fraud decision using MCP tools.

Action matrix — strictly follow this:
- APPROVE        → Log the decision only. Do NOT call any MCP tools.
- FLAG_FOR_REVIEW → Call create_fraud_case(priority="MEDIUM").
- BLOCK_AND_ALERT → First call send_cardholder_alert(alert_type="FRAUD_BLOCK"), then call create_fraud_case(priority="HIGH").

Responsibilities:
1. Read the decision and risk_score from the context.
2. Execute the actions per the action matrix above.
3. Collect results from any MCP tools called.
4. Return a structured JSON summary.

Output ONLY valid JSON with this exact schema:
{
  "action_taken": "<description of action performed>",
  "notifications_sent": ["<list of notification types sent, or empty array if none>"],
  "case_id": "<case ID string, or null if no case created>",
  "timestamp": "<ISO format timestamp>",
  "summary": "<one-sentence summary for the fraud operations team>"
}

Always call all required tools before generating the final JSON output.
Do not include any text outside the JSON block.
"""

print("✅ Prompt templates defined")
for name, prompt in [
    ("FEATURE_EXTRACTOR", FEATURE_EXTRACTOR_SYSTEM_PROMPT),
    ("BEHAVIORAL_ANALYST", BEHAVIORAL_ANALYST_SYSTEM_PROMPT),
    ("FRAUD_CLASSIFIER", FRAUD_CLASSIFIER_SYSTEM_PROMPT),
    ("ACTION_ORCHESTRATOR", ACTION_ORCHESTRATOR_SYSTEM_PROMPT),
]:
    length = len(prompt.strip())
    status = "⚠️  TOO SHORT" if length < 100 else "✔"
    print(f"   {name}: {length} chars {status}")

---
## Section 3: MCP Server Simulation

**MCP (Model Context Protocol)** lets agents call external services via standardized JSON-Schema tool definitions.

| Server | Tool(s) |
|--------|---------|
| `TransactionDBMCPServer` | `get_transaction_history`, `get_account_profile` |
| `MerchantDBMCPServer` | `get_merchant_info` |
| `RiskRulesMCPServer` | `get_fraud_rules` |
| `AlertSystemMCPServer` | `send_cardholder_alert`, `create_fraud_case` |


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# TODO 2a — TransactionDBMCPServer
# ─────────────────────────────────────────────────────────────────────────

class TransactionDBMCPServer:
    """Simulates PayNova's Transaction Database MCP Server"""

    TOOLS = [
        {
            "name": "get_transaction_history",
            "description": "Fetch recent transaction history for a card number from PayNova's transaction database.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "card_number": {"type": "string", "description": "The card number to look up"},
                    "limit": {"type": "integer", "description": "Max number of recent transactions to return (default 10)"}
                },
                "required": ["card_number"]
            }
        },
        {
            "name": "get_account_profile",
            "description": "Fetch account holder profile including credit limit, home location, and average monthly spend.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "card_number": {"type": "string", "description": "The card number to look up"}
                },
                "required": ["card_number"]
            }
        }
    ]

    def get_transaction_history(self, card_number: str, limit: int = 10) -> dict:
        """MCP Tool: Fetch transaction history for a card"""
        if card_number not in TRANSACTION_HISTORY_DB:
            return {"error": "Card not found", "transactions": []}
        data = TRANSACTION_HISTORY_DB[card_number]
        txns = data["recent_transactions"][:limit]
        return {"card_number": card_number, "transaction_count": len(txns), "transactions": txns}

    def get_account_profile(self, card_number: str) -> dict:
        """MCP Tool: Fetch account profile for a card"""
        if card_number not in TRANSACTION_HISTORY_DB:
            return {"error": "Card not found"}
        d = TRANSACTION_HISTORY_DB[card_number]
        return {
            "card_number": card_number,
            "account_holder": d["account_holder"],
            "account_type": d["account_type"],
            "credit_limit": d["credit_limit"],
            "home_location": d["home_location"],
            "avg_monthly_spend": d["avg_monthly_spend"]
        }


txn_db_server = TransactionDBMCPServer()
print(f"TransactionDB tools defined: {len(txn_db_server.TOOLS)}")
assert len(txn_db_server.TOOLS) == 2, "❌ Must define exactly 2 tools in TransactionDBMCPServer.TOOLS"
print("✅ TransactionDBMCPServer OK")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# TODO 2b — MerchantDBMCPServer
# ─────────────────────────────────────────────────────────────────────────

class MerchantDBMCPServer:
    """Simulates PayNova's Merchant Intelligence MCP Server"""

    TOOLS = [
        {
            "name": "get_merchant_info",
            "description": "Fetch merchant risk profile from PayNova's merchant intelligence database.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "merchant_name": {"type": "string", "description": "Name of the merchant to look up"}
                },
                "required": ["merchant_name"]
            }
        }
    ]

    def get_merchant_info(self, merchant_name: str) -> dict:
        """MCP Tool: Fetch merchant risk profile"""
        if merchant_name not in MERCHANT_DB:
            return {
                "merchant_name": merchant_name,
                "risk_level": "unknown",
                "note": "Merchant not found in database — treat as unverified"
            }
        return {"merchant_name": merchant_name, **MERCHANT_DB[merchant_name]}


# ─────────────────────────────────────────────────────────────────────────
# TODO 2c — RiskRulesMCPServer
# ─────────────────────────────────────────────────────────────────────────

class RiskRulesMCPServer:
    """Simulates PayNova's Risk Rules Engine MCP Server"""

    TOOLS = [
        {
            "name": "get_fraud_rules",
            "description": "Fetch PayNova's fraud detection rules. Supports category filter: all, velocity, geo, amount.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "rule_category": {
                        "type": "string",
                        "description": "Category: 'all', 'velocity', 'geo', or 'amount'. Default: 'all'"
                    }
                },
                "required": []
            }
        }
    ]

    def get_fraud_rules(self, rule_category: str = "all") -> dict:
        """MCP Tool: Fetch risk rules"""
        if rule_category == "all":
            return RISK_RULES_DB
        key = f"{rule_category}_rules"
        if key in RISK_RULES_DB:
            return {key: RISK_RULES_DB[key]}
        return {"error": f"Unknown rule category: {rule_category}"}


# ─────────────────────────────────────────────────────────────────────────
# TODO 2d — AlertSystemMCPServer
# ─────────────────────────────────────────────────────────────────────────

class AlertSystemMCPServer:
    """Simulates PayNova's Alert and Case Management MCP Server"""

    TOOLS = [
        {
            "name": "send_cardholder_alert",
            "description": "Send a fraud alert to the cardholder via SMS and Email.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "card_number":     {"type": "string", "description": "The card number of the affected account"},
                    "alert_type":      {"type": "string", "description": "Type of alert, e.g. FRAUD_BLOCK"},
                    "message":         {"type": "string", "description": "Alert message content for the cardholder"},
                    "transaction_id":  {"type": "string", "description": "Related transaction ID (optional)"}
                },
                "required": ["card_number", "alert_type", "message"]
            }
        },
        {
            "name": "create_fraud_case",
            "description": "Create a fraud investigation case in PayNova's case management system.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "card_number":    {"type": "string", "description": "The card number"},
                    "priority":       {"type": "string", "description": "Case priority: HIGH or MEDIUM"},
                    "description":    {"type": "string", "description": "Description of the fraud case"},
                    "transaction_id": {"type": "string", "description": "Related transaction ID (optional)"}
                },
                "required": ["card_number", "priority", "description"]
            }
        }
    ]

    def send_cardholder_alert(self, card_number: str, alert_type: str, message: str,
                               transaction_id: str = None) -> dict:
        """MCP Tool: Send alert to cardholder"""
        recipient = TRANSACTION_HISTORY_DB.get(card_number, {}).get("account_holder", "Unknown Cardholder")
        return {
            "status": "delivered",
            "recipient": recipient,
            "card_number": card_number,
            "alert_type": alert_type,
            "channels": ["SMS", "Email"],
            "message": message,
            "transaction_id": transaction_id,
            "timestamp": datetime.now().isoformat()
        }

    def create_fraud_case(self, card_number: str, priority: str, description: str,
                           transaction_id: str = None) -> dict:
        """MCP Tool: Create a fraud investigation case"""
        case_id = f"PAYNOVA-FRAUD-{datetime.now().strftime('%Y%m%d%H%M%S')}"
        return {
            "case_id": case_id,
            "status": "OPEN",
            "priority": priority,
            "card_number": card_number,
            "description": description,
            "transaction_id": transaction_id,
            "assigned_to": "Fraud Operations Team",
            "created_at": datetime.now().isoformat()
        }


# Instantiate all servers
merchant_db_server  = MerchantDBMCPServer()
risk_rules_server   = RiskRulesMCPServer()
alert_system_server = AlertSystemMCPServer()

# Validation checks
assert len(merchant_db_server.TOOLS)  == 1, "❌ MerchantDBMCPServer needs 1 tool"
assert len(risk_rules_server.TOOLS)   == 1, "❌ RiskRulesMCPServer needs 1 tool"
assert len(alert_system_server.TOOLS) == 2, "❌ AlertSystemMCPServer needs 2 tools"

print("✅ All MCP servers instantiated")
print("   TransactionDB:", [t["name"] for t in txn_db_server.TOOLS])
print("   MerchantDB:   ", [t["name"] for t in merchant_db_server.TOOLS])
print("   RiskRules:    ", [t["name"] for t in risk_rules_server.TOOLS])
print("   AlertSystem:  ", [t["name"] for t in alert_system_server.TOOLS])

---
## Section 4: Agentic Tool-Calling Loop (PROVIDED — read-only)

This is the **engine** that powers all 4 agents. It handles:
1. Sending the task to Claude with available tools
2. Detecting when Claude requests a tool call
3. Dispatching to the correct MCP server handler
4. Feeding the result back to Claude
5. Repeating until Claude produces a final text answer

**You do not modify this function.**


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# PROVIDED — Do not modify
# ─────────────────────────────────────────────────────────────────────────

def run_agent_with_mcp_tools(
    agent_name: str,
    system_prompt: str,
    user_message: str,
    tools: list,
    tool_handlers: dict,
    max_iterations: int = 8,
    verbose: bool = True
) -> str:
    """
    Generic agentic loop with MCP tool support.

    Args:
        agent_name     : Display name for logging
        system_prompt  : Agent system prompt
        user_message   : Task + context for this invocation
        tools          : List of tool definitions from MCP servers
        tool_handlers  : Dict mapping tool_name -> callable handler
        max_iterations : Max tool-use cycles
        verbose        : Print tool calls while running

    Returns:
        Final text response from Claude
    """
    messages = [{"role": "user", "content": user_message}]

    for iteration in range(max_iterations):
        response = client.messages.create(
            model=MODEL,
            max_tokens=2048,
            system=system_prompt,
            tools=tools,
            messages=messages
        )

        if response.stop_reason == "end_turn":
            for block in response.content:
                if hasattr(block, "text"):
                    return block.text
            return "[No text response]"

        elif response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    if verbose:
                        print(f"  🔧 [{agent_name}] MCP: {block.name}({json.dumps(block.input, separators=(',',':'))})")
                    handler = tool_handlers.get(block.name)
                    if handler:
                        result = handler(**block.input)
                    else:
                        result = {"error": f"Tool '{block.name}' not registered"}
                    if verbose:
                        preview = str(result)
                        print(f"     ✔ {preview[:120]}{'...' if len(preview) > 120 else ''}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result)
                    })
            messages.append({"role": "user", "content": tool_results})
        else:
            break

    return "[Max iterations reached]"

print("✅ Agentic tool-calling loop ready")

---
## Section 5: LangGraph State & Agent Nodes

`FraudDetectionState` is a TypedDict that flows through all 4 nodes.
Each node reads from it and returns an updated copy.

Each node must:
1. Extract the current transaction from `state["transaction"]`
2. Build a `user_message` string with the agent's task and relevant context
3. Assemble `tools` and `tool_handlers` from the correct MCP servers
4. Call `run_agent_with_mcp_tools(...)` with the correct prompt and tools
5. Append a log entry to `state["processing_log"]`
6. Return the updated state dict


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# PROVIDED — LangGraph shared state (do not modify)
# ─────────────────────────────────────────────────────────────────────────

class FraudDetectionState(TypedDict):
    transaction:          Dict[str, Any]    # Input transaction dict
    feature_extraction:   Optional[str]     # Agent 1 output
    behavioral_analysis:  Optional[str]     # Agent 2 output
    fraud_classification: Optional[str]     # Agent 3 output
    risk_score:           Optional[float]   # Parsed from Agent 3 output (0.0-1.0)
    decision:             Optional[str]     # APPROVE | FLAG_FOR_REVIEW | BLOCK_AND_ALERT
    action_result:        Optional[str]     # Agent 4 output (final)
    processing_log:       List[str]         # Audit trail

print("✅ FraudDetectionState defined")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# TODO 3a — feature_extractor_node
# ─────────────────────────────────────────────────────────────────────────

def feature_extractor_node(state: FraudDetectionState) -> FraudDetectionState:
    """LangGraph Node 1: Extract transaction features via MCP tools"""
    txn = state["transaction"]
    print(f"\n📊 [Feature Extractor] Analyzing transaction {txn.get('transaction_id')}")

    tools = txn_db_server.TOOLS + merchant_db_server.TOOLS
    tool_handlers = {
        "get_transaction_history": txn_db_server.get_transaction_history,
        "get_account_profile":     txn_db_server.get_account_profile,
        "get_merchant_info":       merchant_db_server.get_merchant_info,
    }

    user_message = f"""Analyze this PayNova transaction for fraud detection.

Transaction JSON:
{json.dumps(txn, indent=2)}

Instructions:
1. Call get_transaction_history(card_number="{txn['card_number']}") to get recent transactions.
2. Call get_account_profile(card_number="{txn['card_number']}") to get account details.
3. Call get_merchant_info(merchant_name="{txn['merchant_name']}") to get merchant risk profile.
4. Return a JSON object with keys: transaction_features, account_baseline, merchant_risk_profile, immediate_flags.
"""

    result = run_agent_with_mcp_tools(
        agent_name="Feature Extractor",
        system_prompt=FEATURE_EXTRACTOR_SYSTEM_PROMPT,
        user_message=user_message,
        tools=tools,
        tool_handlers=tool_handlers
    )

    log = state.get("processing_log", [])
    log.append(f"[{datetime.now().isoformat()}] Feature Extraction completed")

    return {**state, "feature_extraction": result, "processing_log": log}

print("✅ feature_extractor_node defined")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# TODO 3b — behavioral_analyst_node
# ─────────────────────────────────────────────────────────────────────────

def behavioral_analyst_node(state: FraudDetectionState) -> FraudDetectionState:
    """LangGraph Node 2: Analyze behavioral patterns via MCP risk rules"""
    txn = state["transaction"]
    print(f"\n🔍 [Behavioral Analyst] Running anomaly detection")

    tools = risk_rules_server.TOOLS
    tool_handlers = {"get_fraud_rules": risk_rules_server.get_fraud_rules}

    user_message = f"""Perform behavioral anomaly analysis for this PayNova transaction.

Transaction JSON:
{json.dumps(txn, indent=2)}

Feature Extraction Results from previous agent:
{state.get('feature_extraction', 'No feature extraction data available')}

Instructions:
1. Call get_fraud_rules(rule_category="all") to fetch current PayNova thresholds.
2. Score the transaction across all 5 dimensions: Velocity, Geographic, Temporal, Amount, Merchant.
3. Return JSON with keys: velocity_score, geo_score, temporal_score, amount_score, merchant_score, anomalies_detected, reasoning.
"""

    result = run_agent_with_mcp_tools(
        agent_name="Behavioral Analyst",
        system_prompt=BEHAVIORAL_ANALYST_SYSTEM_PROMPT,
        user_message=user_message,
        tools=tools,
        tool_handlers=tool_handlers
    )

    log = state.get("processing_log", [])
    log.append(f"[{datetime.now().isoformat()}] Behavioral Analysis completed")

    return {**state, "behavioral_analysis": result, "processing_log": log}

print("✅ behavioral_analyst_node defined")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# TODO 3c — fraud_classifier_node
# ─────────────────────────────────────────────────────────────────────────

def fraud_classifier_node(state: FraudDetectionState) -> FraudDetectionState:
    """LangGraph Node 3: Produce final fraud classification and risk score"""
    txn = state["transaction"]
    print(f"\n⚖️  [Fraud Classifier] Computing risk score and classification")

    user_message = f"""Classify this PayNova transaction for fraud.

Transaction JSON:
{json.dumps(txn, indent=2)}

Feature Extraction Results:
{state.get('feature_extraction', '')}

Behavioral Analysis Results:
{state.get('behavioral_analysis', '')}

Produce the final classification JSON with keys:
classification, risk_score, confidence, primary_indicators, recommended_action, explanation.
"""

    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=FRAUD_CLASSIFIER_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_message}]
    )
    result = response.content[0].text

    # Parse risk_score and decision from JSON response
    risk_score = 0.5
    decision = "FLAG_FOR_REVIEW"
    try:
        json_match = re.search(r'\{.*\}', result, re.DOTALL)
        if json_match:
            parsed = json.loads(json_match.group())
            risk_score = float(parsed.get("risk_score", 0.5))
            decision = parsed.get("recommended_action", "FLAG_FOR_REVIEW")
    except Exception as e:
        print(f"     ⚠️  JSON parse error: {e} — using defaults")

    print(f"     → risk_score={risk_score:.2f}, decision={decision}")

    log = state.get("processing_log", [])
    log.append(f"[{datetime.now().isoformat()}] Fraud Classification completed: risk_score={risk_score:.2f}, decision={decision}")

    return {
        **state,
        "fraud_classification": result,
        "risk_score": risk_score,
        "decision": decision,
        "processing_log": log
    }

print("✅ fraud_classifier_node defined")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# TODO 3d — action_orchestrator_node
# ─────────────────────────────────────────────────────────────────────────

def action_orchestrator_node(state: FraudDetectionState) -> FraudDetectionState:
    """LangGraph Node 4: Execute protective actions via MCP tools"""
    txn      = state["transaction"]
    decision = state.get("decision", "FLAG_FOR_REVIEW")
    risk_score  = state.get("risk_score", 0.5)
    print(f"\n🚨 [Action Orchestrator] Executing actions for: {decision}")

    tools = alert_system_server.TOOLS
    tool_handlers = {
        "send_cardholder_alert": alert_system_server.send_cardholder_alert,
        "create_fraud_case":     alert_system_server.create_fraud_case,
    }

    user_message = f"""Execute protective actions for this PayNova fraud decision.

Transaction JSON:
{json.dumps(txn, indent=2)}

Decision: {decision}
Risk Score: {risk_score:.2f}
Classification Details:
{state.get('fraud_classification', '')}

Follow the action matrix in your system prompt exactly.
Use the appropriate MCP tools based on the decision.
Return JSON with keys: action_taken, notifications_sent, case_id, timestamp, summary.
"""

    result = run_agent_with_mcp_tools(
        agent_name="Action Orchestrator",
        system_prompt=ACTION_ORCHESTRATOR_SYSTEM_PROMPT,
        user_message=user_message,
        tools=tools,
        tool_handlers=tool_handlers
    )

    log = state.get("processing_log", [])
    log.append(f"[{datetime.now().isoformat()}] Action Orchestration completed: {decision}")

    return {**state, "action_result": result, "processing_log": log}

print("✅ action_orchestrator_node defined")

---
## Section 6: LangGraph Orchestration

Wire all 4 agent nodes into a **StateGraph**.

```
START
  → feature_extractor
  → behavioral_analyst
  → fraud_classifier
  → [conditional routing based on risk_score]
  → action_orchestrator
  → END
```

`route_by_risk` reads `state["risk_score"]` and always routes to `action_orchestrator`
(the orchestrator uses `decision` to determine what tools to call).


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# TODO 4 — Build the LangGraph
# ─────────────────────────────────────────────────────────────────────────

def route_by_risk(state: FraudDetectionState) -> str:
    """Conditional edge: determine next node based on risk score"""
    risk_score = state.get("risk_score", 0.5)
    decision   = state.get("decision", "FLAG_FOR_REVIEW")
    print(f"\n🔀 [Router] risk_score={risk_score:.2f} → routing to: action_orchestrator ({decision})")
    return "action_orchestrator"


def build_fraud_detection_graph():
    """Build and compile the 4-node fraud detection StateGraph"""
    graph = StateGraph(FraudDetectionState)

    # Add all 4 nodes
    graph.add_node("feature_extractor",  feature_extractor_node)
    graph.add_node("behavioral_analyst", behavioral_analyst_node)
    graph.add_node("fraud_classifier",   fraud_classifier_node)
    graph.add_node("action_orchestrator", action_orchestrator_node)

    # Set entry point
    graph.set_entry_point("feature_extractor")

    # Sequential edges
    graph.add_edge("feature_extractor",  "behavioral_analyst")
    graph.add_edge("behavioral_analyst", "fraud_classifier")

    # Conditional edge from fraud_classifier
    graph.add_conditional_edges(
        "fraud_classifier",
        route_by_risk,
        {"action_orchestrator": "action_orchestrator"}
    )

    # Final edge to END
    graph.add_edge("action_orchestrator", END)

    return graph.compile()


# Compile the graph
fraud_detection_app = build_fraud_detection_graph()

print("✅ LangGraph compiled!")
print("Graph flow:")
print("  START → feature_extractor → behavioral_analyst → fraud_classifier")
print("  → [route_by_risk] → action_orchestrator → END")

---
## Section 7: End-to-End Demo

Run both test scenarios through the completed pipeline.

| Scenario | Card | Expected Decision | Why |
|----------|------|------------------|-----|
| 1 — Normal | 4532-0151-1283-0366 | APPROVE | Routine grocery, home city, daytime |
| 2 — Fraud  | 4532-0151-1283-0999 | BLOCK_AND_ALERT | 3 states in 1 hr, 3 AM, high-risk merchants |

**Both scenarios must produce the correct expected decision.**


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# SCENARIO 1 — Normal Transaction — Expected: APPROVE
# ─────────────────────────────────────────────────────────────────────────

normal_transaction = {
    "transaction_id":   "TXN-2026-0404-001",
    "card_number":      "4532-0151-1283-0366",
    "amount":           67.40,
    "merchant_name":    "Whole Foods Market",
    "location":         "San Francisco, CA",
    "timestamp":        "2026-04-04 11:30:00",
    "transaction_type": "in-store",
    "currency":         "USD"
}

# TODO 5a — Initial state for Scenario 1
initial_state_1 = {
    "transaction":          normal_transaction,
    "feature_extraction":   None,
    "behavioral_analysis":  None,
    "fraud_classification": None,
    "risk_score":           None,
    "decision":             None,
    "action_result":        None,
    "processing_log":       []
}

print("=" * 70)
print("SCENARIO 1: Normal Transaction")
print("=" * 70)

result_1 = fraud_detection_app.invoke(initial_state_1)

print("\n" + "=" * 70)
print("SCENARIO 1 RESULTS:")
print(f"  Risk Score : {result_1.get('risk_score', 'N/A')}")
print(f"  Decision   : {result_1.get('decision', 'N/A')}")
expected_1 = "APPROVE"
status_1 = "✅ PASS" if result_1.get("decision") == expected_1 else f"❌ FAIL (expected {expected_1})"
print(f"  Test Result: {status_1}")
print("\nProcessing Log:")
for entry in result_1.get("processing_log", []):
    print(f"  {entry}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# SCENARIO 2 — Fraud Transaction — Expected: BLOCK_AND_ALERT
# ─────────────────────────────────────────────────────────────────────────

fraud_transaction = {
    "transaction_id":   "TXN-2026-0404-009",
    "card_number":      "4532-0151-1283-0999",
    "amount":           9500.00,
    "merchant_name":    "Luxury Boutique NYC",
    "location":         "New York, NY",
    "timestamp":        "2026-04-04 03:12:00",
    "transaction_type": "in-store",
    "currency":         "USD",
    "note":             "Card holder's home is Chicago. 3rd high-value transaction across 3 states in 1 hour."
}

# TODO 5b — Initial state for Scenario 2
initial_state_2 = {
    "transaction":          fraud_transaction,
    "feature_extraction":   None,
    "behavioral_analysis":  None,
    "fraud_classification": None,
    "risk_score":           None,
    "decision":             None,
    "action_result":        None,
    "processing_log":       []
}

print("=" * 70)
print("SCENARIO 2: High-Risk Fraud Transaction (3 states, 3AM, luxury merchants)")
print("=" * 70)

result_2 = fraud_detection_app.invoke(initial_state_2)

print("\n" + "=" * 70)
print("SCENARIO 2 RESULTS:")
print(f"  Risk Score : {result_2.get('risk_score', 'N/A')}")
print(f"  Decision   : {result_2.get('decision', 'N/A')}")
expected_2 = "BLOCK_AND_ALERT"
status_2 = "✅ PASS" if result_2.get("decision") == expected_2 else f"❌ FAIL (expected {expected_2})"
print(f"  Test Result: {status_2}")
print("\nAction Taken:")
action = result_2.get("action_result", "N/A")
print(action[:600] if isinstance(action, str) else "N/A")
print("\nProcessing Log:")
for entry in result_2.get("processing_log", []):
    print(f"  {entry}")

---
## Bonus Exercises (Optional)

These are **optional** exercises. Complete them after the main scenarios pass.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# BONUS 1 (Optional) — Add a 3rd Transaction Scenario (FLAG_FOR_REVIEW)
# ══════════════════════════════════════════════════════════════════════════

# Step 1: Add a 3rd account to TRANSACTION_HISTORY_DB
TRANSACTION_HISTORY_DB["5412-7534-9988-0001"] = {
    "account_holder": "James Rodriguez",
    "account_type": "PayNova Standard",
    "credit_limit": 8000,
    "home_location": "Dallas, TX, USA",
    "avg_monthly_spend": 1200,
    "recent_transactions": [
        {"id": "TXN201", "amount": 55.00, "merchant": "Target", "location": "Dallas, TX", "timestamp": "2026-04-03 15:00:00", "category": "Retail"},
        {"id": "TXN202", "amount": 85.00, "merchant": "Best Buy", "location": "Dallas, TX", "timestamp": "2026-04-02 12:00:00", "category": "Electronics"},
    ]
}

# Step 2: Define a borderline transaction (mild anomalies — slightly unusual amount + unfamiliar location)
borderline_transaction = {
    "transaction_id":   "TXN-2026-0404-050",
    "card_number":      "5412-7534-9988-0001",
    "amount":           800.00,       # larger than usual but not extreme
    "merchant_name":    "Electronics World",  # medium risk merchant
    "location":         "Houston, TX",        # nearby city, ~370 km from Dallas
    "timestamp":        "2026-04-04 21:45:00",# late evening but not suspicious hours
    "transaction_type": "in-store",
    "currency":         "USD"
}

# Step 3: Build initial state and run
initial_state_bonus1 = {
    "transaction":          borderline_transaction,
    "feature_extraction":   None,
    "behavioral_analysis":  None,
    "fraud_classification": None,
    "risk_score":           None,
    "decision":             None,
    "action_result":        None,
    "processing_log":       []
}

print("=" * 70)
print("BONUS 1: Borderline Transaction (expecting FLAG_FOR_REVIEW)")
print("=" * 70)

result_bonus1 = fraud_detection_app.invoke(initial_state_bonus1)
print(f"  Risk Score: {result_bonus1.get('risk_score', 'N/A')}")
print(f"  Decision  : {result_bonus1.get('decision', 'N/A')}")
print("Bonus 1 complete — check if decision is FLAG_FOR_REVIEW")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# BONUS 2 (Optional) — Add a Stolen Card MCP Tool
# ══════════════════════════════════════════════════════════════════════════

STOLEN_CARDS_DB = {
    "4532-0151-1283-0999": {
        "stolen": True,
        "reported_date": "2026-04-03",
        "reason": "Customer reported card lost/stolen"
    }
}

class ExtendedTransactionDBMCPServer(TransactionDBMCPServer):
    TOOLS = TransactionDBMCPServer.TOOLS + [
        {
            "name": "check_card_reported_stolen",
            "description": "Check whether a card has been reported stolen or lost.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "card_number": {"type": "string", "description": "The card number to check"}
                },
                "required": ["card_number"]
            }
        }
    ]

    def check_card_reported_stolen(self, card_number: str) -> dict:
        if card_number in STOLEN_CARDS_DB:
            return {"card_number": card_number, **STOLEN_CARDS_DB[card_number]}
        return {"card_number": card_number, "stolen": False, "reported_date": None, "reason": None}


# Rebind txn_db_server to the extended version
txn_db_server = ExtendedTransactionDBMCPServer()
print(f"Extended TransactionDB tools: {[t['name'] for t in txn_db_server.TOOLS]}")
print("Re-run Scenario 2 to see stolen card flag in feature_extraction output.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# BONUS 3 (Optional) — Parallel Agent Execution (LangGraph Fan-Out)
# ══════════════════════════════════════════════════════════════════════════
# In the main pipeline, behavioral_analyst runs AFTER feature_extractor.
# Here we add a device_fingerprint_analyst that runs IN PARALLEL with
# behavioral_analyst (fan-out), then both feed into fraud_classifier (fan-in).
#
# Key LangGraph 1.x rules for parallel (fan-out) graphs:
#   1. Any field written by multiple parallel nodes MUST use Annotated reducer
#   2. Each node must return ONLY the keys it changes (no {**state, ...})
#   3. Initial state should only contain fields with real values
# ══════════════════════════════════════════════════════════════════════════

from typing import Annotated
import operator


# --- State with Annotated reducer for processing_log ---

class ParallelFraudDetectionState(TypedDict):
    transaction:          Dict[str, Any]
    feature_extraction:   Optional[str]
    behavioral_analysis:  Optional[str]
    device_analysis:      Optional[str]   # NEW — parallel with behavioral_analysis
    fraud_classification: Optional[str]
    risk_score:           Optional[float]
    decision:             Optional[str]
    action_result:        Optional[str]
    processing_log:       Annotated[List[str], operator.add]  # reducer for fan-out


# --- Re-define all nodes with partial returns (required for LangGraph 1.x) ---

def feature_extractor_node(state):
    txn = state['transaction']
    print('[Feature Extractor] Analyzing transaction', txn.get('transaction_id'))
    tools = txn_db_server.TOOLS + merchant_db_server.TOOLS
    tool_handlers = {
        'get_transaction_history': txn_db_server.get_transaction_history,
        'get_account_profile':     txn_db_server.get_account_profile,
        'get_merchant_info':       merchant_db_server.get_merchant_info,
    }
    card = txn['card_number']
    merchant = txn['merchant_name']
    msg = (
        'Analyze this PayNova transaction for fraud detection.\n\n'
        'Transaction JSON:\n' + json.dumps(txn, indent=2) + '\n\n'
        'Call get_transaction_history, get_account_profile, get_merchant_info then '
        'return JSON with keys: transaction_features, account_baseline, merchant_risk_profile, immediate_flags.'
    )
    result = run_agent_with_mcp_tools(
        agent_name='Feature Extractor',
        system_prompt=FEATURE_EXTRACTOR_SYSTEM_PROMPT,
        user_message=msg, tools=tools, tool_handlers=tool_handlers
    )
    return {
        'feature_extraction': result,
        'processing_log': ['[' + datetime.now().isoformat() + '] Feature Extraction completed']
    }


def behavioral_analyst_node(state):
    txn = state['transaction']
    print('[Behavioral Analyst] Running anomaly detection')
    tools = risk_rules_server.TOOLS
    tool_handlers = {'get_fraud_rules': risk_rules_server.get_fraud_rules}
    feat = state.get('feature_extraction', '')
    msg = (
        'Perform behavioral anomaly analysis.\n\n'
        'Transaction JSON:\n' + json.dumps(txn, indent=2) + '\n\n'
        'Feature Extraction Results:\n' + str(feat) + '\n\n'
        'Call get_fraud_rules(rule_category="all") then return JSON with: '
        'velocity_score, geo_score, temporal_score, amount_score, merchant_score, anomalies_detected, reasoning.'
    )
    result = run_agent_with_mcp_tools(
        agent_name='Behavioral Analyst',
        system_prompt=BEHAVIORAL_ANALYST_SYSTEM_PROMPT,
        user_message=msg, tools=tools, tool_handlers=tool_handlers
    )
    return {
        'behavioral_analysis': result,
        'processing_log': ['[' + datetime.now().isoformat() + '] Behavioral Analysis completed']
    }


def fraud_classifier_node(state):
    txn = state['transaction']
    print('[Fraud Classifier] Computing risk score')
    msg = (
        'Classify this PayNova transaction for fraud.\n\n'
        'Transaction JSON:\n' + json.dumps(txn, indent=2) + '\n\n'
        'Feature Extraction:\n' + str(state.get('feature_extraction', '')) + '\n\n'
        'Behavioral Analysis:\n' + str(state.get('behavioral_analysis', '')) + '\n\n'
        'Return JSON with: classification, risk_score, confidence, primary_indicators, recommended_action, explanation.'
    )
    response = client.messages.create(
        model=MODEL, max_tokens=1024,
        system=FRAUD_CLASSIFIER_SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': msg}]
    )
    result = response.content[0].text
    risk_score = 0.5
    decision = 'FLAG_FOR_REVIEW'
    try:
        import re as _re
        m = _re.search(r'{.*}', result, _re.DOTALL)
        if m:
            p = json.loads(m.group())
            risk_score = float(p.get('risk_score', 0.5))
            decision = p.get('recommended_action', 'FLAG_FOR_REVIEW')
    except Exception:
        pass
    print('   -> risk_score=' + str(round(risk_score, 2)) + ', decision=' + str(decision))
    return {
        'fraud_classification': result,
        'risk_score':           risk_score,
        'decision':             decision,
        'processing_log': ['[' + datetime.now().isoformat() + '] Fraud Classification: risk=' + str(round(risk_score, 2)) + ' decision=' + str(decision)]
    }


def action_orchestrator_node(state):
    txn = state['transaction']
    decision = state.get('decision', 'FLAG_FOR_REVIEW')
    risk_score = state.get('risk_score', 0.5)
    print('[Action Orchestrator] Executing:', decision)
    tools = alert_system_server.TOOLS
    tool_handlers = {
        'send_cardholder_alert': alert_system_server.send_cardholder_alert,
        'create_fraud_case':     alert_system_server.create_fraud_case,
    }
    msg = (
        'Execute protective actions for this PayNova fraud decision.\n\n'
        'Transaction JSON:\n' + json.dumps(txn, indent=2) + '\n\n'
        'Decision: ' + str(decision) + '\n'
        'Risk Score: ' + str(round(risk_score, 2)) + '\n'
        'Classification:\n' + str(state.get('fraud_classification', '')) + '\n\n'
        'Use appropriate MCP tools. Return JSON with: action_taken, notifications_sent, case_id, timestamp, summary.'
    )
    result = run_agent_with_mcp_tools(
        agent_name='Action Orchestrator',
        system_prompt=ACTION_ORCHESTRATOR_SYSTEM_PROMPT,
        user_message=msg, tools=tools, tool_handlers=tool_handlers
    )
    return {
        'action_result':  result,
        'processing_log': ['[' + datetime.now().isoformat() + '] Action Orchestration completed: ' + str(decision)]
    }


# --- NEW: Device fingerprint analyst (runs in parallel with behavioral_analyst) ---

def device_fingerprint_analyst_node(state):
    """Parallel Node: Analyze device/IP anomalies (no MCP tools needed)"""
    txn = state['transaction']
    print('[Device Analyst] Analyzing device fingerprint')
    msg = (
        'Analyze device and network signals for this PayNova transaction.\n\n'
        'Transaction:\n' + json.dumps(txn, indent=2) + '\n\n'
        'Assess for device anomalies. If no device fields are present, assign low risk.\n'
        'Return JSON: {"device_risk": "low|medium|high", "flags": [], "reasoning": "..."}'
    )
    response = client.messages.create(
        model=MODEL, max_tokens=512,
        system='You are a PayNova device fingerprint analyst. Output only valid JSON.',
        messages=[{'role': 'user', 'content': msg}]
    )
    return {
        'device_analysis':  response.content[0].text,
        'processing_log':   ['[' + datetime.now().isoformat() + '] Device Analysis completed']
    }


# --- Build parallel graph ---

def build_parallel_fraud_graph():
    """Build graph with fan-out: feature_extractor -> behavioral_analyst + device_analyst in parallel"""
    graph = StateGraph(ParallelFraudDetectionState)

    graph.add_node('feature_extractor',         feature_extractor_node)
    graph.add_node('behavioral_analyst',         behavioral_analyst_node)
    graph.add_node('device_fingerprint_analyst', device_fingerprint_analyst_node)
    graph.add_node('fraud_classifier',           fraud_classifier_node)
    graph.add_node('action_orchestrator',        action_orchestrator_node)

    graph.set_entry_point('feature_extractor')

    # Fan-out: feature_extractor -> BOTH analysts in parallel
    graph.add_edge('feature_extractor', 'behavioral_analyst')
    graph.add_edge('feature_extractor', 'device_fingerprint_analyst')

    # Fan-in: BOTH analysts -> fraud_classifier
    graph.add_edge('behavioral_analyst',         'fraud_classifier')
    graph.add_edge('device_fingerprint_analyst', 'fraud_classifier')

    graph.add_conditional_edges('fraud_classifier', route_by_risk,
                                {'action_orchestrator': 'action_orchestrator'})
    graph.add_edge('action_orchestrator', END)

    return graph.compile()


# --- Run parallel graph ---

print('Building parallel fraud detection graph...')
parallel_app   = build_parallel_fraud_graph()
parallel_state = {
    'transaction':    fraud_transaction,
    'processing_log': []
}

print('Running parallel pipeline on fraud transaction...')
result_parallel = parallel_app.invoke(parallel_state)

print()
print('=' * 60)
print('BONUS 3 RESULTS — Parallel Execution')
print('=' * 60)
print('risk_score     :', result_parallel.get('risk_score'))
print('decision       :', result_parallel.get('decision'))
print('device_analysis:', str(result_parallel.get('device_analysis', ''))[:300])
print('processing_log :', len(result_parallel.get('processing_log', [])), 'entries')
for entry in result_parallel.get('processing_log', []):
    print('  ', entry)

In [ ]:
%pip install streamlit

In [ ]:
# BONUS 4 (Optional) - Streamlit Web App
# Self-contained: writes fraud_app.py to disk then launches Streamlit.
# (c) 2026 Pushpendra Vaishya

import os, sys, subprocess, time, socket, urllib.request

# --- 1. Write fraud_app.py ---
_app = []
_app.append('import os\n')
_app.append('import sys\n')
_app.append('import json\n')
_app.append('import re\n')
_app.append('import asyncio\n')
_app.append('import anthropic\n')
_app.append('import streamlit as st\n')
_app.append('from typing import TypedDict, Optional, List, Dict, Any\n')
_app.append('from langgraph.graph import StateGraph, END\n')
_app.append('from dotenv import load_dotenv\n')
_app.append('from datetime import datetime\n')
_app.append('\n')
_app.append('# Windows event-loop fix (required on Windows Server)\n')
_app.append('if sys.platform == "win32":\n')
_app.append('    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())\n')
_app.append('\n')
_app.append('load_dotenv()\n')
_app.append('api_key = os.environ.get("ANTHROPIC_API_KEY", "")\n')
_app.append('client  = anthropic.Anthropic(api_key=api_key)\n')
_app.append('MODEL   = "claude-opus-4-5"\n')
_app.append('\n')
_app.append('# -----------------------------------------------------------------------------\n')
_app.append('# Section 1 - Mock PayNova Data\n')
_app.append('# -----------------------------------------------------------------------------\n')
_app.append('\n')
_app.append('TRANSACTION_HISTORY_DB = {\n')
_app.append('    "4532-0151-1283-0366": {\n')
_app.append('        "account_holder": "Sarah Johnson",\n')
_app.append('        "account_type": "PayNova Platinum",\n')
_app.append('        "credit_limit": 15000,\n')
_app.append('        "home_location": "San Francisco, CA, USA",\n')
_app.append('        "avg_monthly_spend": 2300,\n')
_app.append('        "recent_transactions": [\n')
_app.append('            {"id": "TXN001", "amount": 45.50,  "merchant": "Whole Foods Market", "location": "San Francisco, CA", "timestamp": "2026-04-03 09:15:00", "category": "Grocery"},\n')
_app.append('            {"id": "TXN002", "amount": 120.00, "merchant": "Amazon.com",         "location": "Online",            "timestamp": "2026-04-02 14:22:00", "category": "E-commerce"},\n')
_app.append('            {"id": "TXN003", "amount": 8.75,   "merchant": "Starbucks",          "location": "San Francisco, CA", "timestamp": "2026-04-01 08:30:00", "category": "Food & Beverage"},\n')
_app.append('            {"id": "TXN004", "amount": 350.00, "merchant": "United Airlines",    "location": "Online",            "timestamp": "2026-03-28 16:45:00", "category": "Travel"},\n')
_app.append('        ]\n')
_app.append('    },\n')
_app.append('    "4532-0151-1283-0999": {\n')
_app.append('        "account_holder": "Mike Thompson",\n')
_app.append('        "account_type": "PayNova Elite",\n')
_app.append('        "credit_limit": 10000,\n')
_app.append('        "home_location": "Chicago, IL, USA",\n')
_app.append('        "avg_monthly_spend": 1800,\n')
_app.append('        "recent_transactions": [\n')
_app.append('            {"id": "TXN101", "amount": 9500.00, "merchant": "Luxury Boutique NYC", "location": "New York, NY",    "timestamp": "2026-04-04 02:15:00", "category": "Luxury Goods"},\n')
_app.append('            {"id": "TXN102", "amount": 4800.00, "merchant": "Electronics World",   "location": "Los Angeles, CA", "timestamp": "2026-04-04 02:48:00", "category": "Electronics"},\n')
_app.append('            {"id": "TXN103", "amount": 3200.00, "merchant": "Diamond Palace",      "location": "Miami, FL",       "timestamp": "2026-04-04 03:10:00", "category": "Jewelry"},\n')
_app.append('        ]\n')
_app.append('    }\n')
_app.append('}\n')
_app.append('\n')
_app.append('MERCHANT_DB = {\n')
_app.append('    "Whole Foods Market":  {"risk_level": "low",    "category": "Grocery",        "avg_transaction": 68,   "fraud_rate": 0.001},\n')
_app.append('    "Amazon.com":          {"risk_level": "low",    "category": "E-commerce",     "avg_transaction": 85,   "fraud_rate": 0.002},\n')
_app.append('    "Starbucks":           {"risk_level": "low",    "category": "Food & Beverage","avg_transaction": 9,    "fraud_rate": 0.0005},\n')
_app.append('    "Luxury Boutique NYC": {"risk_level": "high",   "category": "Luxury Goods",   "avg_transaction": 2500, "fraud_rate": 0.045},\n')
_app.append('    "Electronics World":   {"risk_level": "medium", "category": "Electronics",    "avg_transaction": 450,  "fraud_rate": 0.012},\n')
_app.append('    "Diamond Palace":      {"risk_level": "high",   "category": "Jewelry",        "avg_transaction": 1200, "fraud_rate": 0.038},\n')
_app.append('}\n')
_app.append('\n')
_app.append('RISK_RULES_DB = {\n')
_app.append('    "velocity_rules": {\n')
_app.append('        "max_transactions_per_hour": 5,\n')
_app.append('        "max_amount_per_hour": 5000,\n')
_app.append('        "max_amount_per_day": 15000,\n')
_app.append('    },\n')
_app.append('    "geo_rules": {\n')
_app.append('        "max_distance_from_home_km": 1500,\n')
_app.append('        "suspicious_countries": ["XX", "YY"],\n')
_app.append('        "multiple_country_window_hours": 2,\n')
_app.append('    },\n')
_app.append('    "amount_rules": {\n')
_app.append('        "large_transaction_threshold": 2000,\n')
_app.append('        "unusual_amount_multiplier": 3.0,\n')
_app.append('    }\n')
_app.append('}\n')
_app.append('\n')
_app.append('# -----------------------------------------------------------------------------\n')
_app.append('# Section 2 - System Prompts\n')
_app.append('# -----------------------------------------------------------------------------\n')
_app.append('\n')
_app.append('FEATURE_EXTRACTOR_SYSTEM_PROMPT = (\n')
_app.append('    "You are a specialized PayNova fraud detection agent -- Feature Extractor.\\n"\n')
_app.append('    "Your role is to gather transaction intelligence by calling MCP tools in sequence.\\n\\n"\n')
_app.append('    "Responsibilities:\\n"\n')
_app.append('    "1. Call get_transaction_history(card_number) to retrieve the card\'s recent transactions.\\n"\n')
_app.append('    "2. Call get_account_profile(card_number) to retrieve account holder details.\\n"\n')
_app.append('    "3. Call get_merchant_info(merchant_name) to retrieve the merchant\'s risk profile.\\n"\n')
_app.append('    "4. Synthesize all gathered data into a structured JSON response.\\n\\n"\n')
_app.append('    "Output ONLY valid JSON with this exact schema:\\n"\n')
_app.append("    '{\\n'\n")
_app.append('    \'  "transaction_features": {"transaction_id": "<string>", "amount": <float>, "merchant_name": "<string>", "location": "<string>", "timestamp": "<string>", "transaction_type": "<string>", "currency": "<string>"},\\n\'\n')
_app.append('    \'  "account_baseline": {"account_holder": "<string>", "account_type": "<string>", "credit_limit": <float>, "home_location": "<string>", "avg_monthly_spend": <float>, "recent_transaction_count": <int>, "recent_transactions": []},\\n\'\n')
_app.append('    \'  "merchant_risk_profile": {"merchant_name": "<string>", "risk_level": "<string>", "category": "<string>", "avg_transaction": <float>, "fraud_rate": <float>},\\n\'\n')
_app.append('    \'  "immediate_flags": ["<list any obvious red flags, or empty array>"]\\n\'\n')
_app.append('    "}\\n\\n"\n')
_app.append('    "Think step by step. Call all 3 tools before producing the final JSON output.\\n"\n')
_app.append('    "Do not include any text outside the JSON block."\n')
_app.append(')\n')
_app.append('\n')
_app.append('BEHAVIORAL_ANALYST_SYSTEM_PROMPT = (\n')
_app.append('    "You are a specialized PayNova fraud detection agent -- Behavioral Analyst.\\n"\n')
_app.append('    "Your role is to detect anomalous behavioral patterns by scoring the transaction across 5 risk dimensions.\\n\\n"\n')
_app.append('    "Responsibilities:\\n"\n')
_app.append('    "1. Call get_fraud_rules(rule_category=\\"all\\") to fetch PayNova\'s current thresholds.\\n"\n')
_app.append('    "2. Analyze the transaction across 5 dimensions using the rules and feature context provided.\\n"\n')
_app.append('    "3. Assign a risk score (0.0-1.0) for each dimension.\\n\\n"\n')
_app.append('    "Scoring guide (each dimension scored 0.0-1.0):\\n"\n')
_app.append('    "- Velocity: 0.8-1.0 if transactions exceed max_transactions_per_hour or max_amount_per_hour; 0.3-0.5 approaching limits; 0.0-0.2 normal\\n"\n')
_app.append('    "- Geographic: 0.8-1.0 if >1500 km from home_location or suspicious country; 0.4-0.7 moderately distant; 0.0-0.2 near home\\n"\n')
_app.append('    "- Temporal: 0.8-1.0 for 12AM-5AM; 0.4-0.6 late night (10PM-12AM); 0.0-0.2 business hours\\n"\n')
_app.append('    "- Amount: 0.8-1.0 if > large_transaction_threshold AND > 3x (avg_monthly_spend/10); 0.4-0.6 moderately large; 0.0-0.2 normal\\n"\n')
_app.append('    "- Merchant: 0.8-1.0 high risk; 0.3-0.6 medium; 0.0-0.2 low; 0.5 unknown\\n\\n"\n')
_app.append('    "Output ONLY valid JSON with this exact schema:\\n"\n')
_app.append("    '{\\n'\n")
_app.append('    \'  "velocity_score": <float 0.0-1.0>,\\n\'\n')
_app.append('    \'  "geo_score": <float 0.0-1.0>,\\n\'\n')
_app.append('    \'  "temporal_score": <float 0.0-1.0>,\\n\'\n')
_app.append('    \'  "amount_score": <float 0.0-1.0>,\\n\'\n')
_app.append('    \'  "merchant_score": <float 0.0-1.0>,\\n\'\n')
_app.append('    \'  "anomalies_detected": ["<list of detected anomalies as strings>"],\\n\'\n')
_app.append('    \'  "reasoning": "<step-by-step explanation of each score>"\\n\'\n')
_app.append('    "}\\n\\n"\n')
_app.append('    "Think step by step. Call get_fraud_rules first, then score each dimension carefully.\\n"\n')
_app.append('    "Do not include any text outside the JSON block."\n')
_app.append(')\n')
_app.append('\n')
_app.append('FRAUD_CLASSIFIER_SYSTEM_PROMPT = (\n')
_app.append('    "You are a specialized PayNova fraud detection agent -- Fraud Classifier.\\n"\n')
_app.append('    "Your role is to produce a definitive fraud classification by synthesizing Feature Extraction and Behavioral Analysis outputs.\\n"\n')
_app.append('    "You do NOT call any tools -- you reason purely from the context provided.\\n\\n"\n')
_app.append('    "Classification tiers based on composite risk_score:\\n"\n')
_app.append('    "- LEGITIMATE: risk_score < 0.4  -> recommended_action: APPROVE\\n"\n')
_app.append('    "- SUSPICIOUS:  0.4 <= risk_score < 0.8 -> recommended_action: FLAG_FOR_REVIEW\\n"\n')
_app.append('    "- FRAUDULENT:  risk_score >= 0.8 -> recommended_action: BLOCK_AND_ALERT\\n\\n"\n')
_app.append('    "Compute risk_score using this weighted formula:\\n"\n')
_app.append('    "  risk_score = (velocity_score * 0.25) + (geo_score * 0.20) + (temporal_score * 0.20) + (amount_score * 0.20) + (merchant_score * 0.15)\\n\\n"\n')
_app.append('    "Then adjust upward (add 0.05-0.15) if the Feature Extractor raised multiple immediate_flags.\\n"\n')
_app.append('    "Cap risk_score at 1.0. Round to 2 decimal places.\\n\\n"\n')
_app.append('    "Output ONLY valid JSON with this exact schema:\\n"\n')
_app.append("    '{\\n'\n")
_app.append('    \'  "classification": "LEGITIMATE" | "SUSPICIOUS" | "FRAUDULENT",\\n\'\n')
_app.append('    \'  "risk_score": <float 0.0-1.0>,\\n\'\n')
_app.append('    \'  "confidence": <float 0.0-1.0>,\\n\'\n')
_app.append('    \'  "primary_indicators": ["<top 3 reasons driving the score>"],\\n\'\n')
_app.append('    \'  "recommended_action": "APPROVE" | "FLAG_FOR_REVIEW" | "BLOCK_AND_ALERT",\\n\'\n')
_app.append('    \'  "explanation": "<2-4 sentence plain-language explanation for the fraud analyst>"\\n\'\n')
_app.append('    "}\\n\\n"\n')
_app.append('    "Be decisive. When in doubt between tiers, use the higher-risk classification.\\n"\n')
_app.append('    "Do not include any text outside the JSON block."\n')
_app.append(')\n')
_app.append('\n')
_app.append('ACTION_ORCHESTRATOR_SYSTEM_PROMPT = (\n')
_app.append('    "You are a specialized PayNova fraud detection agent -- Action Orchestrator.\\n"\n')
_app.append('    "Your role is to execute the correct protective actions based on the fraud decision using MCP tools.\\n\\n"\n')
_app.append('    "Action matrix -- strictly follow this:\\n"\n')
_app.append('    "- APPROVE        -> Log the decision only. Do NOT call any MCP tools.\\n"\n')
_app.append('    "- FLAG_FOR_REVIEW -> Call create_fraud_case(priority=\\"MEDIUM\\").\\n"\n')
_app.append('    "- BLOCK_AND_ALERT -> First call send_cardholder_alert(alert_type=\\"FRAUD_BLOCK\\"), then call create_fraud_case(priority=\\"HIGH\\").\\n\\n"\n')
_app.append('    "Output ONLY valid JSON with this exact schema:\\n"\n')
_app.append("    '{\\n'\n")
_app.append('    \'  "action_taken": "<description of action performed>",\\n\'\n')
_app.append('    \'  "notifications_sent": ["<list of notification types sent, or empty array if none>"],\\n\'\n')
_app.append('    \'  "case_id": "<case ID string, or null if no case created>",\\n\'\n')
_app.append('    \'  "timestamp": "<ISO format timestamp>",\\n\'\n')
_app.append('    \'  "summary": "<one-sentence summary for the fraud operations team>"\\n\'\n')
_app.append('    "}\\n\\n"\n')
_app.append('    "Always call all required tools before generating the final JSON output.\\n"\n')
_app.append('    "Do not include any text outside the JSON block."\n')
_app.append(')\n')
_app.append('\n')
_app.append('# -----------------------------------------------------------------------------\n')
_app.append('# Section 3 - MCP Server Classes\n')
_app.append('# -----------------------------------------------------------------------------\n')
_app.append('\n')
_app.append('class TransactionDBMCPServer:\n')
_app.append('    TOOLS = [\n')
_app.append('        {\n')
_app.append('            "name": "get_transaction_history",\n')
_app.append('            "description": "Fetch recent transaction history for a card number from PayNova\'s transaction database.",\n')
_app.append('            "input_schema": {\n')
_app.append('                "type": "object",\n')
_app.append('                "properties": {\n')
_app.append('                    "card_number": {"type": "string", "description": "The card number to look up"},\n')
_app.append('                    "limit": {"type": "integer", "description": "Max number of recent transactions (default 10)"}\n')
_app.append('                },\n')
_app.append('                "required": ["card_number"]\n')
_app.append('            }\n')
_app.append('        },\n')
_app.append('        {\n')
_app.append('            "name": "get_account_profile",\n')
_app.append('            "description": "Fetch account holder profile including credit limit, home location, and average monthly spend.",\n')
_app.append('            "input_schema": {\n')
_app.append('                "type": "object",\n')
_app.append('                "properties": {\n')
_app.append('                    "card_number": {"type": "string", "description": "The card number to look up"}\n')
_app.append('                },\n')
_app.append('                "required": ["card_number"]\n')
_app.append('            }\n')
_app.append('        }\n')
_app.append('    ]\n')
_app.append('\n')
_app.append('    def get_transaction_history(self, card_number: str, limit: int = 10) -> dict:\n')
_app.append('        if card_number not in TRANSACTION_HISTORY_DB:\n')
_app.append('            return {"error": "Card not found", "transactions": []}\n')
_app.append('        data = TRANSACTION_HISTORY_DB[card_number]\n')
_app.append('        txns = data["recent_transactions"][:limit]\n')
_app.append('        return {"card_number": card_number, "transaction_count": len(txns), "transactions": txns}\n')
_app.append('\n')
_app.append('    def get_account_profile(self, card_number: str) -> dict:\n')
_app.append('        if card_number not in TRANSACTION_HISTORY_DB:\n')
_app.append('            return {"error": "Card not found"}\n')
_app.append('        d = TRANSACTION_HISTORY_DB[card_number]\n')
_app.append('        return {\n')
_app.append('            "card_number": card_number,\n')
_app.append('            "account_holder": d["account_holder"],\n')
_app.append('            "account_type": d["account_type"],\n')
_app.append('            "credit_limit": d["credit_limit"],\n')
_app.append('            "home_location": d["home_location"],\n')
_app.append('            "avg_monthly_spend": d["avg_monthly_spend"]\n')
_app.append('        }\n')
_app.append('\n')
_app.append('\n')
_app.append('class MerchantDBMCPServer:\n')
_app.append('    TOOLS = [\n')
_app.append('        {\n')
_app.append('            "name": "get_merchant_info",\n')
_app.append('            "description": "Fetch merchant risk profile from PayNova\'s merchant intelligence database.",\n')
_app.append('            "input_schema": {\n')
_app.append('                "type": "object",\n')
_app.append('                "properties": {\n')
_app.append('                    "merchant_name": {"type": "string", "description": "Name of the merchant to look up"}\n')
_app.append('                },\n')
_app.append('                "required": ["merchant_name"]\n')
_app.append('            }\n')
_app.append('        }\n')
_app.append('    ]\n')
_app.append('\n')
_app.append('    def get_merchant_info(self, merchant_name: str) -> dict:\n')
_app.append('        if merchant_name not in MERCHANT_DB:\n')
_app.append('            return {"merchant_name": merchant_name, "risk_level": "unknown", "note": "Merchant not in database"}\n')
_app.append('        return {"merchant_name": merchant_name, **MERCHANT_DB[merchant_name]}\n')
_app.append('\n')
_app.append('\n')
_app.append('class RiskRulesMCPServer:\n')
_app.append('    TOOLS = [\n')
_app.append('        {\n')
_app.append('            "name": "get_fraud_rules",\n')
_app.append('            "description": "Fetch PayNova\'s fraud detection rules. Supports category: all, velocity, geo, amount.",\n')
_app.append('            "input_schema": {\n')
_app.append('                "type": "object",\n')
_app.append('                "properties": {\n')
_app.append('                    "rule_category": {"type": "string", "description": "Category: \'all\', \'velocity\', \'geo\', or \'amount\'. Default: \'all\'"}\n')
_app.append('                },\n')
_app.append('                "required": []\n')
_app.append('            }\n')
_app.append('        }\n')
_app.append('    ]\n')
_app.append('\n')
_app.append('    def get_fraud_rules(self, rule_category: str = "all") -> dict:\n')
_app.append('        if rule_category == "all":\n')
_app.append('            return RISK_RULES_DB\n')
_app.append('        key = f"{rule_category}_rules"\n')
_app.append('        if key in RISK_RULES_DB:\n')
_app.append('            return {key: RISK_RULES_DB[key]}\n')
_app.append('        return {"error": f"Unknown rule category: {rule_category}"}\n')
_app.append('\n')
_app.append('\n')
_app.append('class AlertSystemMCPServer:\n')
_app.append('    TOOLS = [\n')
_app.append('        {\n')
_app.append('            "name": "send_cardholder_alert",\n')
_app.append('            "description": "Send a fraud alert to the cardholder via SMS and Email.",\n')
_app.append('            "input_schema": {\n')
_app.append('                "type": "object",\n')
_app.append('                "properties": {\n')
_app.append('                    "card_number":    {"type": "string"},\n')
_app.append('                    "alert_type":     {"type": "string"},\n')
_app.append('                    "message":        {"type": "string"},\n')
_app.append('                    "transaction_id": {"type": "string"}\n')
_app.append('                },\n')
_app.append('                "required": ["card_number", "alert_type", "message"]\n')
_app.append('            }\n')
_app.append('        },\n')
_app.append('        {\n')
_app.append('            "name": "create_fraud_case",\n')
_app.append('            "description": "Create a fraud investigation case in PayNova\'s case management system.",\n')
_app.append('            "input_schema": {\n')
_app.append('                "type": "object",\n')
_app.append('                "properties": {\n')
_app.append('                    "card_number":    {"type": "string"},\n')
_app.append('                    "priority":       {"type": "string"},\n')
_app.append('                    "description":    {"type": "string"},\n')
_app.append('                    "transaction_id": {"type": "string"}\n')
_app.append('                },\n')
_app.append('                "required": ["card_number", "priority", "description"]\n')
_app.append('            }\n')
_app.append('        }\n')
_app.append('    ]\n')
_app.append('\n')
_app.append('    def send_cardholder_alert(self, card_number: str, alert_type: str, message: str,\n')
_app.append('                               transaction_id: str = None) -> dict:\n')
_app.append('        recipient = TRANSACTION_HISTORY_DB.get(card_number, {}).get("account_holder", "Unknown Cardholder")\n')
_app.append('        return {\n')
_app.append('            "status": "delivered",\n')
_app.append('            "recipient": recipient,\n')
_app.append('            "card_number": card_number,\n')
_app.append('            "alert_type": alert_type,\n')
_app.append('            "channels": ["SMS", "Email"],\n')
_app.append('            "message": message,\n')
_app.append('            "transaction_id": transaction_id,\n')
_app.append('            "timestamp": datetime.now().isoformat()\n')
_app.append('        }\n')
_app.append('\n')
_app.append('    def create_fraud_case(self, card_number: str, priority: str, description: str,\n')
_app.append('                           transaction_id: str = None) -> dict:\n')
_app.append('        case_id = f"PAYNOVA-FRAUD-{datetime.now().strftime(\'%Y%m%d%H%M%S\')}"\n')
_app.append('        return {\n')
_app.append('            "case_id": case_id,\n')
_app.append('            "status": "OPEN",\n')
_app.append('            "priority": priority,\n')
_app.append('            "card_number": card_number,\n')
_app.append('            "description": description,\n')
_app.append('            "transaction_id": transaction_id,\n')
_app.append('            "assigned_to": "Fraud Operations Team",\n')
_app.append('            "created_at": datetime.now().isoformat()\n')
_app.append('        }\n')
_app.append('\n')
_app.append('\n')
_app.append('txn_db_server       = TransactionDBMCPServer()\n')
_app.append('merchant_db_server  = MerchantDBMCPServer()\n')
_app.append('risk_rules_server   = RiskRulesMCPServer()\n')
_app.append('alert_system_server = AlertSystemMCPServer()\n')
_app.append('\n')
_app.append('# -----------------------------------------------------------------------------\n')
_app.append('# Section 4 - Agentic Tool-Calling Loop\n')
_app.append('# -----------------------------------------------------------------------------\n')
_app.append('\n')
_app.append('def run_agent_with_mcp_tools(\n')
_app.append('    agent_name: str,\n')
_app.append('    system_prompt: str,\n')
_app.append('    user_message: str,\n')
_app.append('    tools: list,\n')
_app.append('    tool_handlers: dict,\n')
_app.append('    max_iterations: int = 8,\n')
_app.append('    verbose: bool = True\n')
_app.append(') -> str:\n')
_app.append('    messages = [{"role": "user", "content": user_message}]\n')
_app.append('\n')
_app.append('    for iteration in range(max_iterations):\n')
_app.append('        response = client.messages.create(\n')
_app.append('            model=MODEL,\n')
_app.append('            max_tokens=2048,\n')
_app.append('            system=system_prompt,\n')
_app.append('            tools=tools,\n')
_app.append('            messages=messages\n')
_app.append('        )\n')
_app.append('\n')
_app.append('        if response.stop_reason == "end_turn":\n')
_app.append('            for block in response.content:\n')
_app.append('                if hasattr(block, "text"):\n')
_app.append('                    return block.text\n')
_app.append('            return "[No text response]"\n')
_app.append('\n')
_app.append('        elif response.stop_reason == "tool_use":\n')
_app.append('            messages.append({"role": "assistant", "content": response.content})\n')
_app.append('            tool_results = []\n')
_app.append('            for block in response.content:\n')
_app.append('                if block.type == "tool_use":\n')
_app.append('                    if verbose:\n')
_app.append('                        print(f"  [MCP] [{agent_name}] {block.name}({json.dumps(block.input, separators=(\',\',\':\'))})")\n')
_app.append('                    handler = tool_handlers.get(block.name)\n')
_app.append('                    result  = handler(**block.input) if handler else {"error": f"Tool \'{block.name}\' not registered"}\n')
_app.append('                    tool_results.append({\n')
_app.append('                        "type": "tool_result",\n')
_app.append('                        "tool_use_id": block.id,\n')
_app.append('                        "content": json.dumps(result)\n')
_app.append('                    })\n')
_app.append('            messages.append({"role": "user", "content": tool_results})\n')
_app.append('        else:\n')
_app.append('            break\n')
_app.append('\n')
_app.append('    return "[Max iterations reached]"\n')
_app.append('\n')
_app.append('# -----------------------------------------------------------------------------\n')
_app.append('# Section 5 - LangGraph State & Agent Nodes\n')
_app.append('# -----------------------------------------------------------------------------\n')
_app.append('\n')
_app.append('class FraudDetectionState(TypedDict):\n')
_app.append('    transaction:          Dict[str, Any]\n')
_app.append('    feature_extraction:   Optional[str]\n')
_app.append('    behavioral_analysis:  Optional[str]\n')
_app.append('    fraud_classification: Optional[str]\n')
_app.append('    risk_score:           Optional[float]\n')
_app.append('    decision:             Optional[str]\n')
_app.append('    action_result:        Optional[str]\n')
_app.append('    processing_log:       List[str]\n')
_app.append('\n')
_app.append('\n')
_app.append('def feature_extractor_node(state: FraudDetectionState) -> FraudDetectionState:\n')
_app.append('    txn = state["transaction"]\n')
_app.append('    print(f"[Feature Extractor] Analyzing {txn.get(\'transaction_id\')}")\n')
_app.append('    tools = txn_db_server.TOOLS + merchant_db_server.TOOLS\n')
_app.append('    tool_handlers = {\n')
_app.append('        "get_transaction_history": txn_db_server.get_transaction_history,\n')
_app.append('        "get_account_profile":     txn_db_server.get_account_profile,\n')
_app.append('        "get_merchant_info":       merchant_db_server.get_merchant_info,\n')
_app.append('    }\n')
_app.append('    user_message = (\n')
_app.append('        f"Analyze this PayNova transaction for fraud detection.\\n\\n"\n')
_app.append('        f"Transaction JSON:\\n{json.dumps(txn, indent=2)}\\n\\n"\n')
_app.append('        f"Instructions:\\n"\n')
_app.append('        f"1. Call get_transaction_history(card_number=\\"{txn[\'card_number\']}\\")\\n"\n')
_app.append('        f"2. Call get_account_profile(card_number=\\"{txn[\'card_number\']}\\")\\n"\n')
_app.append('        f"3. Call get_merchant_info(merchant_name=\\"{txn[\'merchant_name\']}\\")\\n"\n')
_app.append('        f"4. Return JSON: transaction_features, account_baseline, merchant_risk_profile, immediate_flags."\n')
_app.append('    )\n')
_app.append('    result = run_agent_with_mcp_tools("Feature Extractor", FEATURE_EXTRACTOR_SYSTEM_PROMPT,\n')
_app.append('                                       user_message, tools, tool_handlers)\n')
_app.append('    log = state.get("processing_log", [])\n')
_app.append('    log.append(f"[{datetime.now().isoformat()}] Feature Extraction completed")\n')
_app.append('    return {**state, "feature_extraction": result, "processing_log": log}\n')
_app.append('\n')
_app.append('\n')
_app.append('def behavioral_analyst_node(state: FraudDetectionState) -> FraudDetectionState:\n')
_app.append('    txn = state["transaction"]\n')
_app.append('    print(f"[Behavioral Analyst] Running anomaly detection")\n')
_app.append('    tools = risk_rules_server.TOOLS\n')
_app.append('    tool_handlers = {"get_fraud_rules": risk_rules_server.get_fraud_rules}\n')
_app.append('    user_message = (\n')
_app.append('        f"Perform behavioral anomaly analysis for this PayNova transaction.\\n\\n"\n')
_app.append('        f"Transaction JSON:\\n{json.dumps(txn, indent=2)}\\n\\n"\n')
_app.append('        f"Feature Extraction Results:\\n{state.get(\'feature_extraction\', \'No data\')}\\n\\n"\n')
_app.append('        f"1. Call get_fraud_rules(rule_category=\\"all\\")\\n"\n')
_app.append('        f"2. Score all 5 dimensions.\\n"\n')
_app.append('        f"3. Return JSON: velocity_score, geo_score, temporal_score, amount_score, merchant_score, anomalies_detected, reasoning."\n')
_app.append('    )\n')
_app.append('    result = run_agent_with_mcp_tools("Behavioral Analyst", BEHAVIORAL_ANALYST_SYSTEM_PROMPT,\n')
_app.append('                                       user_message, tools, tool_handlers)\n')
_app.append('    log = state.get("processing_log", [])\n')
_app.append('    log.append(f"[{datetime.now().isoformat()}] Behavioral Analysis completed")\n')
_app.append('    return {**state, "behavioral_analysis": result, "processing_log": log}\n')
_app.append('\n')
_app.append('\n')
_app.append('def fraud_classifier_node(state: FraudDetectionState) -> FraudDetectionState:\n')
_app.append('    txn = state["transaction"]\n')
_app.append('    print(f"[Fraud Classifier] Computing risk score")\n')
_app.append('    user_message = (\n')
_app.append('        f"Classify this PayNova transaction for fraud.\\n\\n"\n')
_app.append('        f"Transaction JSON:\\n{json.dumps(txn, indent=2)}\\n\\n"\n')
_app.append('        f"Feature Extraction Results:\\n{state.get(\'feature_extraction\', \'\')}\\n\\n"\n')
_app.append('        f"Behavioral Analysis Results:\\n{state.get(\'behavioral_analysis\', \'\')}\\n\\n"\n')
_app.append('        f"Return JSON: classification, risk_score, confidence, primary_indicators, recommended_action, explanation."\n')
_app.append('    )\n')
_app.append('    response = client.messages.create(\n')
_app.append('        model=MODEL, max_tokens=1024,\n')
_app.append('        system=FRAUD_CLASSIFIER_SYSTEM_PROMPT,\n')
_app.append('        messages=[{"role": "user", "content": user_message}]\n')
_app.append('    )\n')
_app.append('    result = response.content[0].text\n')
_app.append('    risk_score = 0.5\n')
_app.append('    decision   = "FLAG_FOR_REVIEW"\n')
_app.append('    try:\n')
_app.append("        m = re.search(r'\\{.*\\}', result, re.DOTALL)\n")
_app.append('        if m:\n')
_app.append('            parsed     = json.loads(m.group())\n')
_app.append('            risk_score = float(parsed.get("risk_score", 0.5))\n')
_app.append('            decision   = parsed.get("recommended_action", "FLAG_FOR_REVIEW")\n')
_app.append('    except Exception:\n')
_app.append('        pass\n')
_app.append('    print(f"  -> risk_score={risk_score:.2f}, decision={decision}")\n')
_app.append('    log = state.get("processing_log", [])\n')
_app.append('    log.append(f"[{datetime.now().isoformat()}] Fraud Classification: risk_score={risk_score:.2f}, decision={decision}")\n')
_app.append('    return {**state, "fraud_classification": result, "risk_score": risk_score, "decision": decision, "processing_log": log}\n')
_app.append('\n')
_app.append('\n')
_app.append('def action_orchestrator_node(state: FraudDetectionState) -> FraudDetectionState:\n')
_app.append('    txn        = state["transaction"]\n')
_app.append('    decision   = state.get("decision", "FLAG_FOR_REVIEW")\n')
_app.append('    risk_score = state.get("risk_score", 0.5)\n')
_app.append('    print(f"[Action Orchestrator] Executing: {decision}")\n')
_app.append('    tools = alert_system_server.TOOLS\n')
_app.append('    tool_handlers = {\n')
_app.append('        "send_cardholder_alert": alert_system_server.send_cardholder_alert,\n')
_app.append('        "create_fraud_case":     alert_system_server.create_fraud_case,\n')
_app.append('    }\n')
_app.append('    user_message = (\n')
_app.append('        f"Execute protective actions for this PayNova fraud decision.\\n\\n"\n')
_app.append('        f"Transaction JSON:\\n{json.dumps(txn, indent=2)}\\n\\n"\n')
_app.append('        f"Decision: {decision}\\nRisk Score: {risk_score:.2f}\\n"\n')
_app.append('        f"Classification:\\n{state.get(\'fraud_classification\', \'\')}\\n\\n"\n')
_app.append('        f"Follow the action matrix. Return JSON: action_taken, notifications_sent, case_id, timestamp, summary."\n')
_app.append('    )\n')
_app.append('    result = run_agent_with_mcp_tools("Action Orchestrator", ACTION_ORCHESTRATOR_SYSTEM_PROMPT,\n')
_app.append('                                       user_message, tools, tool_handlers)\n')
_app.append('    log = state.get("processing_log", [])\n')
_app.append('    log.append(f"[{datetime.now().isoformat()}] Action Orchestration completed: {decision}")\n')
_app.append('    return {**state, "action_result": result, "processing_log": log}\n')
_app.append('\n')
_app.append('# -----------------------------------------------------------------------------\n')
_app.append('# Section 6 - LangGraph Orchestration\n')
_app.append('# -----------------------------------------------------------------------------\n')
_app.append('\n')
_app.append('def route_by_risk(state: FraudDetectionState) -> str:\n')
_app.append('    return "action_orchestrator"\n')
_app.append('\n')
_app.append('\n')
_app.append('def build_fraud_detection_graph():\n')
_app.append('    graph = StateGraph(FraudDetectionState)\n')
_app.append('    graph.add_node("feature_extractor",   feature_extractor_node)\n')
_app.append('    graph.add_node("behavioral_analyst",  behavioral_analyst_node)\n')
_app.append('    graph.add_node("fraud_classifier",    fraud_classifier_node)\n')
_app.append('    graph.add_node("action_orchestrator", action_orchestrator_node)\n')
_app.append('    graph.set_entry_point("feature_extractor")\n')
_app.append('    graph.add_edge("feature_extractor",  "behavioral_analyst")\n')
_app.append('    graph.add_edge("behavioral_analyst", "fraud_classifier")\n')
_app.append('    graph.add_conditional_edges("fraud_classifier", route_by_risk, {"action_orchestrator": "action_orchestrator"})\n')
_app.append('    graph.add_edge("action_orchestrator", END)\n')
_app.append('    return graph.compile()\n')
_app.append('\n')
_app.append('# =============================================================================\n')
_app.append('# Streamlit UI  -  PayNova Fraud Detection Dashboard\n')
_app.append('# =============================================================================\n')
_app.append('\n')
_app.append('st.set_page_config(\n')
_app.append('    page_title="PayNova | Fraud Detection",\n')
_app.append('    page_icon="https://img.icons8.com/fluency/48/shield.png",\n')
_app.append('    layout="wide",\n')
_app.append('    initial_sidebar_state="collapsed",\n')
_app.append(')\n')
_app.append('\n')
_app.append('# ---------------------------------------------------------------------------\n')
_app.append('# Global CSS\n')
_app.append('# ---------------------------------------------------------------------------\n')
_app.append('st.markdown("""\n')
_app.append('<style>\n')
_app.append("@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');\n")
_app.append('\n')
_app.append('/* ── Base ── */\n')
_app.append('html, body, [data-testid="stAppViewContainer"] {\n')
_app.append("    font-family: 'Inter', sans-serif;\n")
_app.append('    background: #060d18;\n')
_app.append('    color: #cdd9e5;\n')
_app.append('}\n')
_app.append('[data-testid="stHeader"]          { background: transparent !important; }\n')
_app.append('[data-testid="stToolbar"]         { display: none; }\n')
_app.append('[data-testid="stDecoration"]      { display: none; }\n')
_app.append('[data-testid="stSidebar"]         { background: #0a1628; border-right: 1px solid #1a2d45; }\n')
_app.append('section[data-testid="stMain"]     { padding-top: 0; }\n')
_app.append('\n')
_app.append('/* ── Top navbar ── */\n')
_app.append('.navbar {\n')
_app.append('    display: flex;\n')
_app.append('    align-items: center;\n')
_app.append('    justify-content: space-between;\n')
_app.append('    background: linear-gradient(90deg, #050c18 0%, #0a1628 100%);\n')
_app.append('    border-bottom: 1px solid #1a2d45;\n')
_app.append('    padding: 14px 32px;\n')
_app.append('    margin: -1rem -1rem 0 -1rem;\n')
_app.append('}\n')
_app.append('.navbar-brand {\n')
_app.append('    display: flex; align-items: center; gap: 12px;\n')
_app.append('}\n')
_app.append('.navbar-logo {\n')
_app.append('    width: 36px; height: 36px;\n')
_app.append('    background: linear-gradient(135deg, #1565c0, #00b4d8);\n')
_app.append('    border-radius: 8px;\n')
_app.append('    display: flex; align-items: center; justify-content: center;\n')
_app.append('    font-size: 18px; font-weight: 700; color: white;\n')
_app.append('    flex-shrink: 0;\n')
_app.append('}\n')
_app.append('.navbar-title {\n')
_app.append('    font-size: 1.15rem; font-weight: 700; color: #e2e8f0;\n')
_app.append('    letter-spacing: 0.3px;\n')
_app.append('}\n')
_app.append('.navbar-subtitle {\n')
_app.append('    font-size: 0.72rem; color: #64748b; letter-spacing: 0.5px;\n')
_app.append('    text-transform: uppercase;\n')
_app.append('}\n')
_app.append('.navbar-status {\n')
_app.append('    display: flex; align-items: center; gap: 8px;\n')
_app.append('    font-size: 0.78rem; color: #4ade80; font-weight: 500;\n')
_app.append('}\n')
_app.append('.status-dot {\n')
_app.append('    width: 8px; height: 8px; border-radius: 50%;\n')
_app.append('    background: #4ade80;\n')
_app.append('    box-shadow: 0 0 6px #4ade80;\n')
_app.append('    animation: pulse 2s infinite;\n')
_app.append('}\n')
_app.append('@keyframes pulse {\n')
_app.append('    0%, 100% { opacity: 1; }\n')
_app.append('    50%       { opacity: 0.4; }\n')
_app.append('}\n')
_app.append('\n')
_app.append('/* ── Stat cards ── */\n')
_app.append('.stat-row {\n')
_app.append('    display: flex; gap: 16px; margin: 20px 0 8px 0;\n')
_app.append('}\n')
_app.append('.stat-card {\n')
_app.append('    flex: 1;\n')
_app.append('    background: #0d1f35;\n')
_app.append('    border: 1px solid #1a2d45;\n')
_app.append('    border-radius: 10px;\n')
_app.append('    padding: 14px 18px;\n')
_app.append('    display: flex; flex-direction: column; gap: 4px;\n')
_app.append('}\n')
_app.append('.stat-label  { font-size: 0.72rem; color: #64748b; text-transform: uppercase; letter-spacing: 0.6px; }\n')
_app.append('.stat-value  { font-size: 1.4rem;  color: #e2e8f0; font-weight: 700; }\n')
_app.append('.stat-sub    { font-size: 0.72rem; color: #4ade80; }\n')
_app.append('.stat-sub.warn { color: #fb923c; }\n')
_app.append('\n')
_app.append('/* ── Section headings ── */\n')
_app.append('.section-heading {\n')
_app.append('    font-size: 0.8rem; font-weight: 600; color: #64748b;\n')
_app.append('    text-transform: uppercase; letter-spacing: 0.8px;\n')
_app.append('    margin: 20px 0 10px 0;\n')
_app.append('    display: flex; align-items: center; gap: 8px;\n')
_app.append('}\n')
_app.append('.section-heading::after {\n')
_app.append("    content: ''; flex: 1;\n")
_app.append('    height: 1px; background: #1a2d45;\n')
_app.append('}\n')
_app.append('\n')
_app.append('/* ── Form card ── */\n')
_app.append('.form-card {\n')
_app.append('    background: #0d1f35;\n')
_app.append('    border: 1px solid #1a2d45;\n')
_app.append('    border-radius: 12px;\n')
_app.append('    padding: 22px 24px;\n')
_app.append('}\n')
_app.append('\n')
_app.append('/* ── Inputs ── */\n')
_app.append('[data-testid="stTextInput"] input,\n')
_app.append('[data-testid="stNumberInput"] input {\n')
_app.append('    background: #060d18 !important;\n')
_app.append('    color: #cdd9e5 !important;\n')
_app.append('    border: 1px solid #1e3a5f !important;\n')
_app.append('    border-radius: 7px !important;\n')
_app.append('    font-size: 0.88rem !important;\n')
_app.append('}\n')
_app.append('[data-testid="stTextInput"] input:focus,\n')
_app.append('[data-testid="stNumberInput"] input:focus {\n')
_app.append('    border-color: #0288d1 !important;\n')
_app.append('    box-shadow: 0 0 0 2px rgba(2,136,209,0.2) !important;\n')
_app.append('}\n')
_app.append('[data-testid="stTextInput"] label,\n')
_app.append('[data-testid="stNumberInput"] label,\n')
_app.append('[data-testid="stSelectbox"] label {\n')
_app.append('    color: #94a3b8 !important; font-size: 0.8rem !important; font-weight: 500 !important;\n')
_app.append('}\n')
_app.append('[data-testid="stSelectbox"] > div > div {\n')
_app.append('    background: #060d18 !important; color: #cdd9e5 !important;\n')
_app.append('    border: 1px solid #1e3a5f !important; border-radius: 7px !important;\n')
_app.append('}\n')
_app.append('\n')
_app.append('/* ── Run button ── */\n')
_app.append('[data-testid="stButton"] > button[kind="primary"] {\n')
_app.append('    background: linear-gradient(90deg, #0d47a1 0%, #0288d1 100%) !important;\n')
_app.append('    color: white !important;\n')
_app.append('    border: none !important;\n')
_app.append('    border-radius: 8px !important;\n')
_app.append('    font-size: 0.9rem !important;\n')
_app.append('    font-weight: 600 !important;\n')
_app.append('    letter-spacing: 0.4px !important;\n')
_app.append('    padding: 0.6rem 1.2rem !important;\n')
_app.append('    transition: all 0.2s !important;\n')
_app.append('    box-shadow: 0 4px 14px rgba(2,136,209,0.35) !important;\n')
_app.append('}\n')
_app.append('[data-testid="stButton"] > button[kind="primary"]:hover {\n')
_app.append('    transform: translateY(-1px) !important;\n')
_app.append('    box-shadow: 0 6px 20px rgba(2,136,209,0.5) !important;\n')
_app.append('}\n')
_app.append('\n')
_app.append('/* ── Decision banner ── */\n')
_app.append('.decision-banner {\n')
_app.append('    border-radius: 10px;\n')
_app.append('    padding: 18px 22px;\n')
_app.append('    display: flex; align-items: center; gap: 16px;\n')
_app.append('    margin-bottom: 16px;\n')
_app.append('}\n')
_app.append('.decision-banner.approve {\n')
_app.append('    background: rgba(74,222,128,0.08);\n')
_app.append('    border: 1px solid rgba(74,222,128,0.35);\n')
_app.append('}\n')
_app.append('.decision-banner.review {\n')
_app.append('    background: rgba(251,146,60,0.08);\n')
_app.append('    border: 1px solid rgba(251,146,60,0.35);\n')
_app.append('}\n')
_app.append('.decision-banner.block {\n')
_app.append('    background: rgba(248,113,113,0.08);\n')
_app.append('    border: 1px solid rgba(248,113,113,0.35);\n')
_app.append('}\n')
_app.append('.decision-icon { font-size: 2rem; }\n')
_app.append('.decision-text h2 { margin: 0; font-size: 1.1rem; font-weight: 700; }\n')
_app.append('.decision-text p  { margin: 2px 0 0 0; font-size: 0.8rem; opacity: 0.8; }\n')
_app.append('.decision-approve { color: #4ade80; }\n')
_app.append('.decision-review  { color: #fb923c; }\n')
_app.append('.decision-block   { color: #f87171; }\n')
_app.append('\n')
_app.append('/* ── Risk bar ── */\n')
_app.append('.risk-bar-wrap {\n')
_app.append('    background: #0a1628;\n')
_app.append('    border: 1px solid #1a2d45;\n')
_app.append('    border-radius: 10px;\n')
_app.append('    padding: 16px 20px;\n')
_app.append('    margin-bottom: 16px;\n')
_app.append('}\n')
_app.append('.risk-bar-label {\n')
_app.append('    display: flex; justify-content: space-between;\n')
_app.append('    font-size: 0.78rem; color: #94a3b8; margin-bottom: 8px;\n')
_app.append('}\n')
_app.append('.risk-bar-track {\n')
_app.append('    height: 8px; background: #1a2d45; border-radius: 100px; overflow: hidden;\n')
_app.append('}\n')
_app.append('.risk-bar-fill {\n')
_app.append('    height: 100%; border-radius: 100px;\n')
_app.append('    transition: width 0.6s ease;\n')
_app.append('}\n')
_app.append('\n')
_app.append('/* ── Metric chips ── */\n')
_app.append('.chip-row { display: flex; flex-wrap: wrap; gap: 8px; margin-bottom: 16px; }\n')
_app.append('.chip {\n')
_app.append('    background: #0d1f35;\n')
_app.append('    border: 1px solid #1a2d45;\n')
_app.append('    border-radius: 20px;\n')
_app.append('    padding: 4px 12px;\n')
_app.append('    font-size: 0.75rem; color: #94a3b8;\n')
_app.append('}\n')
_app.append('.chip span { color: #e2e8f0; font-weight: 600; }\n')
_app.append('\n')
_app.append('/* ── Pipeline steps ── */\n')
_app.append('.pipeline {\n')
_app.append('    display: flex; align-items: center; gap: 0;\n')
_app.append('    margin: 14px 0 20px 0; overflow-x: auto;\n')
_app.append('}\n')
_app.append('.pipe-step {\n')
_app.append('    flex: 1; text-align: center;\n')
_app.append('    background: #0d1f35;\n')
_app.append('    border: 1px solid #1a2d45;\n')
_app.append('    border-radius: 8px;\n')
_app.append('    padding: 10px 6px;\n')
_app.append('    font-size: 0.7rem; color: #64748b;\n')
_app.append('    position: relative;\n')
_app.append('}\n')
_app.append('.pipe-step.done {\n')
_app.append('    border-color: #0288d1;\n')
_app.append('    color: #7dd3fc;\n')
_app.append('    background: rgba(2,136,209,0.06);\n')
_app.append('}\n')
_app.append('.pipe-step .pipe-icon { font-size: 1.1rem; display: block; margin-bottom: 3px; }\n')
_app.append('.pipe-arrow {\n')
_app.append('    color: #1a2d45; font-size: 1.1rem; padding: 0 2px; flex-shrink: 0;\n')
_app.append('}\n')
_app.append('.pipe-arrow.done { color: #0288d1; }\n')
_app.append('\n')
_app.append('/* ── Agent expanders ── */\n')
_app.append('details summary {\n')
_app.append('    background: #0d1f35 !important;\n')
_app.append('    border: 1px solid #1a2d45 !important;\n')
_app.append('    border-radius: 8px !important;\n')
_app.append('    color: #94a3b8 !important;\n')
_app.append('    font-size: 0.82rem !important;\n')
_app.append('    font-weight: 500 !important;\n')
_app.append('}\n')
_app.append('[data-testid="stExpander"] {\n')
_app.append('    background: #0d1f35 !important;\n')
_app.append('    border: 1px solid #1a2d45 !important;\n')
_app.append('    border-radius: 8px !important;\n')
_app.append('}\n')
_app.append('\n')
_app.append('/* ── Code / log ── */\n')
_app.append('[data-testid="stCode"] > div {\n')
_app.append('    background: #020810 !important;\n')
_app.append('    border: 1px solid #1a2d45 !important;\n')
_app.append('    border-radius: 8px !important;\n')
_app.append('    font-size: 0.75rem !important;\n')
_app.append('}\n')
_app.append('\n')
_app.append('/* ── Footer ── */\n')
_app.append('.footer {\n')
_app.append('    margin-top: 40px;\n')
_app.append('    padding: 18px 0;\n')
_app.append('    border-top: 1px solid #1a2d45;\n')
_app.append('    text-align: center;\n')
_app.append('    color: #334155;\n')
_app.append('    font-size: 0.75rem;\n')
_app.append('    letter-spacing: 0.3px;\n')
_app.append('}\n')
_app.append('.footer a { color: #475569; text-decoration: none; }\n')
_app.append('</style>\n')
_app.append('""", unsafe_allow_html=True)\n')
_app.append('\n')
_app.append('# ---------------------------------------------------------------------------\n')
_app.append('# Top Navbar\n')
_app.append('# ---------------------------------------------------------------------------\n')
_app.append('st.markdown("""\n')
_app.append('<div class="navbar">\n')
_app.append('    <div class="navbar-brand">\n')
_app.append('        <div class="navbar-logo">P</div>\n')
_app.append('        <div>\n')
_app.append('            <div class="navbar-title">PayNova &nbsp; Fraud Intelligence</div>\n')
_app.append('            <div class="navbar-subtitle">Real-Time Multi-Agent Detection System</div>\n')
_app.append('        </div>\n')
_app.append('    </div>\n')
_app.append('    <div class="navbar-status">\n')
_app.append('        <div class="status-dot"></div>\n')
_app.append('        System Online &nbsp;&nbsp;|&nbsp;&nbsp; Claude claude-opus-4-5 &nbsp;&nbsp;|&nbsp;&nbsp; LangGraph v0.2\n')
_app.append('    </div>\n')
_app.append('</div>\n')
_app.append('""", unsafe_allow_html=True)\n')
_app.append('\n')
_app.append('# ---------------------------------------------------------------------------\n')
_app.append('# KPI stat bar\n')
_app.append('# ---------------------------------------------------------------------------\n')
_app.append('st.markdown("""\n')
_app.append('<div class="stat-row">\n')
_app.append('    <div class="stat-card">\n')
_app.append('        <div class="stat-label">Engine</div>\n')
_app.append('        <div class="stat-value">4-Agent</div>\n')
_app.append('        <div class="stat-sub">Sequential Pipeline</div>\n')
_app.append('    </div>\n')
_app.append('    <div class="stat-card">\n')
_app.append('        <div class="stat-label">MCP Servers</div>\n')
_app.append('        <div class="stat-value">4 Active</div>\n')
_app.append('        <div class="stat-sub">TxnDB / MerchantDB / Rules / Alerts</div>\n')
_app.append('    </div>\n')
_app.append('    <div class="stat-card">\n')
_app.append('        <div class="stat-label">Model</div>\n')
_app.append('        <div class="stat-value">Opus 4.5</div>\n')
_app.append('        <div class="stat-sub">Anthropic Claude</div>\n')
_app.append('    </div>\n')
_app.append('    <div class="stat-card">\n')
_app.append('        <div class="stat-label">Decision Tiers</div>\n')
_app.append('        <div class="stat-value">3 Levels</div>\n')
_app.append('        <div class="stat-sub">Approve / Review / Block</div>\n')
_app.append('    </div>\n')
_app.append('</div>\n')
_app.append('""", unsafe_allow_html=True)\n')
_app.append('\n')
_app.append('# ---------------------------------------------------------------------------\n')
_app.append('# Preset data\n')
_app.append('# ---------------------------------------------------------------------------\n')
_app.append('PRESETS = {\n')
_app.append('    "-- Select a preset or enter manually --": {},\n')
_app.append('    "Scenario 1: Normal Transaction (Expected: APPROVE)": {\n')
_app.append('        "card_number": "4532-0151-1283-0366",\n')
_app.append('        "amount": 67.40,\n')
_app.append('        "merchant_name": "Whole Foods Market",\n')
_app.append('        "location": "San Francisco, CA",\n')
_app.append('        "timestamp": "2026-04-04 11:30:00",\n')
_app.append('    },\n')
_app.append('    "Scenario 2: Fraudulent Transaction (Expected: BLOCK)": {\n')
_app.append('        "card_number": "4532-0151-1283-0999",\n')
_app.append('        "amount": 9500.00,\n')
_app.append('        "merchant_name": "Luxury Boutique NYC",\n')
_app.append('        "location": "New York, NY",\n')
_app.append('        "timestamp": "2026-04-04 03:12:00",\n')
_app.append('    },\n')
_app.append('}\n')
_app.append('\n')
_app.append('# ---------------------------------------------------------------------------\n')
_app.append('# Main layout\n')
_app.append('# ---------------------------------------------------------------------------\n')
_app.append('col_form, col_results = st.columns([5, 7], gap="large")\n')
_app.append('\n')
_app.append('# ── LEFT: Transaction Input ──────────────────────────────────────────────────\n')
_app.append('with col_form:\n')
_app.append('    st.markdown(\'<div class="section-heading">Transaction Input</div>\', unsafe_allow_html=True)\n')
_app.append('\n')
_app.append('    st.markdown(\'<div class="form-card">\', unsafe_allow_html=True)\n')
_app.append('    preset = st.selectbox("Load test scenario", list(PRESETS.keys()), label_visibility="visible")\n')
_app.append('    p      = PRESETS[preset]\n')
_app.append('\n')
_app.append('    st.markdown("<br>", unsafe_allow_html=True)\n')
_app.append('    card_number   = st.text_input("Card Number",    value=p.get("card_number", ""),    placeholder="e.g. 4532-0151-1283-0366")\n')
_app.append('    amount        = st.number_input("Amount (USD)", value=float(p.get("amount", 0.0)), min_value=0.0, step=0.01, format="%.2f")\n')
_app.append('\n')
_app.append('    c1, c2 = st.columns(2)\n')
_app.append('    with c1:\n')
_app.append('        merchant_name = st.text_input("Merchant Name", value=p.get("merchant_name", ""), placeholder="e.g. Amazon.com")\n')
_app.append('    with c2:\n')
_app.append('        location = st.text_input("Location", value=p.get("location", ""), placeholder="e.g. San Francisco, CA")\n')
_app.append('\n')
_app.append('    timestamp = st.text_input("Timestamp", value=p.get("timestamp", datetime.now().strftime("%Y-%m-%d %H:%M:%S")))\n')
_app.append('    st.markdown("</div>", unsafe_allow_html=True)\n')
_app.append('\n')
_app.append('    st.markdown("<br>", unsafe_allow_html=True)\n')
_app.append('    run_btn = st.button("Run Fraud Detection Pipeline", type="primary", use_container_width=True)\n')
_app.append('\n')
_app.append('    # Pipeline diagram (always visible)\n')
_app.append('    st.markdown(\'<div class="section-heading">Agent Pipeline</div>\', unsafe_allow_html=True)\n')
_app.append('    st.markdown("""\n')
_app.append('    <div class="pipeline">\n')
_app.append('        <div class="pipe-step">\n')
_app.append('            <span class="pipe-icon">🔍</span>\n')
_app.append('            Feature<br>Extractor\n')
_app.append('        </div>\n')
_app.append('        <div class="pipe-arrow">›</div>\n')
_app.append('        <div class="pipe-step">\n')
_app.append('            <span class="pipe-icon">📊</span>\n')
_app.append('            Behavioral<br>Analyst\n')
_app.append('        </div>\n')
_app.append('        <div class="pipe-arrow">›</div>\n')
_app.append('        <div class="pipe-step">\n')
_app.append('            <span class="pipe-icon">🧠</span>\n')
_app.append('            Fraud<br>Classifier\n')
_app.append('        </div>\n')
_app.append('        <div class="pipe-arrow">›</div>\n')
_app.append('        <div class="pipe-step">\n')
_app.append('            <span class="pipe-icon">⚡</span>\n')
_app.append('            Action<br>Orchestrator\n')
_app.append('        </div>\n')
_app.append('    </div>\n')
_app.append('    """, unsafe_allow_html=True)\n')
_app.append('\n')
_app.append('# ── RIGHT: Results ───────────────────────────────────────────────────────────\n')
_app.append('with col_results:\n')
_app.append('    st.markdown(\'<div class="section-heading">Analysis Results</div>\', unsafe_allow_html=True)\n')
_app.append('\n')
_app.append('    if not run_btn:\n')
_app.append('        st.markdown("""\n')
_app.append('        <div style="background:#0d1f35;border:1px dashed #1a2d45;border-radius:12px;\n')
_app.append('                    padding:48px 24px;text-align:center;color:#334155;">\n')
_app.append('            <div style="font-size:2.5rem;margin-bottom:12px;">🛡️</div>\n')
_app.append('            <div style="font-size:1rem;font-weight:600;color:#475569;">Awaiting Transaction</div>\n')
_app.append('            <div style="font-size:0.82rem;margin-top:6px;">\n')
_app.append('                Select a preset scenario or fill in transaction details,<br>\n')
_app.append('                then click <strong style="color:#0288d1;">Run Fraud Detection Pipeline</strong>.\n')
_app.append('            </div>\n')
_app.append('        </div>\n')
_app.append('        """, unsafe_allow_html=True)\n')
_app.append('\n')
_app.append('    else:\n')
_app.append('        if not card_number or not merchant_name:\n')
_app.append('            st.error("Card Number and Merchant Name are required.")\n')
_app.append('        else:\n')
_app.append('            txn = {\n')
_app.append('                "transaction_id":   "TXN-" + datetime.now().strftime("%Y%m%d-%H%M%S"),\n')
_app.append('                "card_number":      card_number,\n')
_app.append('                "amount":           amount,\n')
_app.append('                "merchant_name":    merchant_name,\n')
_app.append('                "location":         location,\n')
_app.append('                "timestamp":        timestamp,\n')
_app.append('                "transaction_type": "in-store",\n')
_app.append('                "currency":         "USD",\n')
_app.append('            }\n')
_app.append('\n')
_app.append('            with st.spinner("Running 4-agent fraud detection pipeline..."):\n')
_app.append('                try:\n')
_app.append('                    graph_app = build_fraud_detection_graph()\n')
_app.append('                    result    = graph_app.invoke({"transaction": txn, "processing_log": []})\n')
_app.append('                except Exception as e:\n')
_app.append('                    st.error(f"Pipeline error: {e}")\n')
_app.append('                    result = {}\n')
_app.append('\n')
_app.append('            risk = result.get("risk_score") or 0.0\n')
_app.append('            dec  = result.get("decision")   or "N/A"\n')
_app.append('\n')
_app.append('            # Decision config\n')
_app.append('            DEC_CFG = {\n')
_app.append('                "APPROVE":         ("approve", "✅", "decision-approve", "Transaction Approved",         "Low risk — no action required."),\n')
_app.append('                "FLAG_FOR_REVIEW": ("review",  "⚠️", "decision-review",  "Flagged for Manual Review",    "Medium risk — fraud case opened for investigation."),\n')
_app.append('                "BLOCK_AND_ALERT": ("block",   "🚫", "decision-block",   "Transaction Blocked & Alerted","High risk — cardholder notified, fraud case escalated."),\n')
_app.append('            }\n')
_app.append('            cls, icon, txt_cls, title, subtitle = DEC_CFG.get(\n')
_app.append('                dec, ("review", "❓", "decision-review", dec, "Unknown decision"))\n')
_app.append('\n')
_app.append('            # ── Decision banner ──\n')
_app.append('            st.markdown(f"""\n')
_app.append('            <div class="decision-banner {cls}">\n')
_app.append('                <div class="decision-icon">{icon}</div>\n')
_app.append('                <div class="decision-text {txt_cls}">\n')
_app.append('                    <h2>{title}</h2>\n')
_app.append('                    <p>{subtitle}</p>\n')
_app.append('                </div>\n')
_app.append('            </div>\n')
_app.append('            """, unsafe_allow_html=True)\n')
_app.append('\n')
_app.append('            # ── Risk score bar ──\n')
_app.append('            bar_pct  = int(risk * 100)\n')
_app.append('            if risk < 0.4:\n')
_app.append('                bar_col = "#4ade80"\n')
_app.append('            elif risk < 0.8:\n')
_app.append('                bar_col = "#fb923c"\n')
_app.append('            else:\n')
_app.append('                bar_col = "#f87171"\n')
_app.append('\n')
_app.append('            st.markdown(f"""\n')
_app.append('            <div class="risk-bar-wrap">\n')
_app.append('                <div class="risk-bar-label">\n')
_app.append('                    <span>Risk Score</span>\n')
_app.append('                    <span style="color:{bar_col};font-weight:700;font-size:1rem;">{risk:.2f} / 1.00</span>\n')
_app.append('                </div>\n')
_app.append('                <div class="risk-bar-track">\n')
_app.append('                    <div class="risk-bar-fill" style="width:{bar_pct}%;background:{bar_col};"></div>\n')
_app.append('                </div>\n')
_app.append('                <div style="display:flex;justify-content:space-between;font-size:0.68rem;color:#334155;margin-top:5px;">\n')
_app.append('                    <span>0.0 — Low</span><span>0.4 — Medium</span><span>0.8 — High</span>\n')
_app.append('                </div>\n')
_app.append('            </div>\n')
_app.append('            """, unsafe_allow_html=True)\n')
_app.append('\n')
_app.append('            # ── Transaction chips ──\n')
_app.append('            st.markdown(f"""\n')
_app.append('            <div class="chip-row">\n')
_app.append('                <div class="chip">ID &nbsp;<span>{txn["transaction_id"]}</span></div>\n')
_app.append('                <div class="chip">Amount &nbsp;<span>${amount:,.2f}</span></div>\n')
_app.append('                <div class="chip">Merchant &nbsp;<span>{merchant_name}</span></div>\n')
_app.append('                <div class="chip">Location &nbsp;<span>{location or "N/A"}</span></div>\n')
_app.append('            </div>\n')
_app.append('            """, unsafe_allow_html=True)\n')
_app.append('\n')
_app.append('            # ── Agent output expanders ──\n')
_app.append('            st.markdown(\'<div class="section-heading">Agent Outputs</div>\', unsafe_allow_html=True)\n')
_app.append('\n')
_app.append('            AGENT_META = [\n')
_app.append('                ("🔍  Feature Extraction",   "feature_extraction",   "Data gathered from TransactionDB, MerchantDB"),\n')
_app.append('                ("📊  Behavioral Analysis",  "behavioral_analysis",  "5-dimension anomaly scoring"),\n')
_app.append('                ("🧠  Fraud Classification", "fraud_classification", "Weighted risk score + classification"),\n')
_app.append('                ("⚡  Action Taken",          "action_result",        "MCP actions executed by orchestrator"),\n')
_app.append('            ]\n')
_app.append('            for lbl, key, hint in AGENT_META:\n')
_app.append('                with st.expander(f"{lbl}  —  {hint}"):\n')
_app.append('                    raw = result.get(key) or "-"\n')
_app.append('                    # Try to pretty-print JSON\n')
_app.append('                    try:\n')
_app.append("                        m = re.search(r'\\{.*\\}', raw, re.DOTALL)\n")
_app.append('                        if m:\n')
_app.append('                            st.json(json.loads(m.group()))\n')
_app.append('                        else:\n')
_app.append('                            st.text(raw)\n')
_app.append('                    except Exception:\n')
_app.append('                        st.text(raw)\n')
_app.append('\n')
_app.append('            # ── Processing log ──\n')
_app.append('            st.markdown(\'<div class="section-heading">Processing Log</div>\', unsafe_allow_html=True)\n')
_app.append('            log_entries = result.get("processing_log", ["No log entries."])\n')
_app.append('            st.code("\\n".join(log_entries), language=None)\n')
_app.append('\n')
_app.append('# ---------------------------------------------------------------------------\n')
_app.append('# Footer\n')
_app.append('# ---------------------------------------------------------------------------\n')
_app.append('st.markdown("""\n')
_app.append('<div class="footer">\n')
_app.append('    &copy; 2026 &nbsp;<strong style="color:#475569;">Pushpendra Vaishya</strong> &nbsp;|&nbsp;\n')
_app.append('    PayNova Fraud Intelligence Platform &nbsp;|&nbsp;\n')
_app.append('    Powered by <strong style="color:#475569;">Claude claude-opus-4-5</strong> (Anthropic) + LangGraph &nbsp;|&nbsp;\n')
_app.append('    All rights reserved\n')
_app.append('</div>\n')
_app.append('""", unsafe_allow_html=True)\n')

with open('fraud_app.py', 'w', encoding='utf-8') as _f:
    _f.writelines(_app)
print(f'[OK] fraud_app.py written ({len(_app)} lines)')

# --- 2. Find a free port ---
def _free_port():
    s = socket.socket(); s.bind(('', 0)); p = s.getsockname()[1]; s.close(); return p

port = _free_port()

# --- 3. Launch Streamlit as a separate process ---
proc = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'fraud_app.py',
     '--server.port', str(port),
     '--server.headless', 'true',
     '--browser.gatherUsageStats', 'false'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(f'[OK] Streamlit started (PID {proc.pid})')

# --- 4. Wait until ready (up to 15s) ---
ready = False
for _ in range(30):
    time.sleep(0.5)
    try:
        urllib.request.urlopen('http://localhost:' + str(port), timeout=1)
        ready = True; break
    except Exception:
        pass

# --- 5. Print the URL ---
print()
print('=' * 55)
print('Streamlit is RUNNING!' if ready else 'Streamlit is starting...')
print()
print('  Open in browser --> http://localhost:' + str(port))
print()
print('Press the STOP (interrupt) button in Jupyter to stop.')
print('=' * 55)

---
## Discussion Questions — Required

**Answer all 5 questions below. Each answer should be 3–6 sentences.**

---

Sure, here are the answers rewritten in the same style:

---

**Q1. MCP Architecture**
Why is MCP a better architecture than calling Python functions directly inside an agent?
What operational advantages does it provide in a production payment network like PayNova?

> MCP enhances the separation of concerns by decoupling agents from their data sources via a standardized JSON-Schema protocol. This means services like TransactionDB, MerchantDB, and RiskRulesDB can be independently versioned, secured, and maintained. In PayNova’s production environment, this isolation allows different teams — for instance, the database team for TransactionDB and the risk team for RiskRulesDB — to manage their respective services without disrupting the agent layer. MCP also facilitates strict access control: each tool enforces authentication, rate limits, and audit logging before any data is provided to the agents, which is essential for PCI-DSS compliance. Furthermore, MCP’s network-transparent design allows the system to function as distributed microservices without modifying the agents themselves.

---

**Q2. Agent Decomposition**
We used 4 agents instead of 1 powerful agent. Could a single agent do the whole job?
What are the trade-offs of the multi-agent approach?

> A single agent could theoretically handle all tasks (data gathering, scoring behavior, classifying risk, and decision-making). However, using 4 specialized agents brings advantages like clearer responsibility, easier debugging, and independent scalability. Each agent has a narrow focus with a defined system prompt, making its output more predictable and easier to validate. If an error occurs, pinpointing the exact agent causing it is simpler, compared to debugging a monolithic approach. The trade-offs include added latency from each additional API round-trip and increased orchestration complexity, but for fraud detection, the accountability, auditability, and ease of validation outweigh these costs.

---

**Q3. Prompt Engineering**
Compare your `FRAUD_CLASSIFIER_SYSTEM_PROMPT` with your `BEHAVIORAL_ANALYST_SYSTEM_PROMPT`.
Which relies more on structured output? Which relies more on reasoning?

> The `FRAUD_CLASSIFIER_SYSTEM_PROMPT` relies heavily on structured output: it specifies a precise JSON schema, sets numeric thresholds, and includes a deterministic formula (`risk_score = velocity*0.25 + ...`), leaving little room for reasoning. On the other hand, the `BEHAVIORAL_ANALYST_SYSTEM_PROMPT` demands more reasoning: the agent needs to use contextual judgment to translate raw data (like transaction amount, location, and time) into 0–1 scores, often requiring domain-specific inference. As a result, the classifier’s output is predictable and deterministic due to its rigid structure, whereas the analyst’s output may vary because it depends on the agent’s reasoning process. To reduce variability in production, setting the analyst’s `temperature` to 0 and providing a few-shot example would help stabilize results.

---

**Q4. LangGraph State Design**
`FraudDetectionState` uses `Optional[str]` for most fields.
How would you redesign it to handle 10,000 transactions per second?

> Handling 10,000 transactions per second would require a significant redesign to optimize memory and performance. First, `Optional[str]` fields storing raw LLM output should be replaced with compact Pydantic models that only store essential scalars (e.g., scores, flags) rather than large text outputs. Next, intermediate results should be stored in an external database (such as Redis or DynamoDB), using transaction IDs as keys to reduce in-memory data load. `Processing_log` should be removed from the state entirely, instead streaming logs directly to a backend service. Additionally, fields like `feature_extraction` and `behavioral_analysis` can be marked as ephemeral, ensuring they are cleared once the classifier consumes them, minimizing memory usage during peak load.

---

**Q5. Production Readiness**
Identify 3 specific changes you would make before deploying this PoC to PayNova's production environment (consider PCI-DSS compliance, latency, cost, and reliability).

> **1. PCI-DSS Compliance:** The PoC currently sends raw cardholder data (e.g., card numbers and names) into LLM prompts, which violates PCI-DSS guidelines. In production, cardholder data should be tokenized before being sent to any LLM, and servers should return tokenized identifiers instead of raw card data.
> **2. Latency:** The current design involves chaining 4 Claude API calls, resulting in response times of 8–20 seconds, which exceeds the desired 100ms SLA. To address this, we can run Feature Extractor and Behavioral Analyst concurrently via LangGraph fan-out, pre-fetch MCP data, and enable prompt caching to reduce turnaround times.
> **3. Reliability and Cost:** The system lacks retries, fallback logic, and cost control measures. In production, we need circuit breakers for each Claude API call, a simple fallback classifier (like a rule-based scoring function), and request-level timeouts to maintain SLA during latency spikes and to manage API costs effectively.

---

---
*End of Final Case Study — Good luck!*
*Submit as:* **FinalCS_FraudDetection_Pushpendra_Vaishya_20552596.ipynb**
